# Pandas by doing: UN General Debate speeches and economic data

This notebook is built around tasks. Every concept follows the same rhythm:

1. A short explanation and one or two small examples (read, run).
2. A task that asks you to do the same thing on **different** countries, years or variables (write).
3. A hidden solution underneath, with the output you should get on the practice corpus.

The examples use Switzerland, Germany and the USA. The tasks use Brazil, India, Kenya, Japan, Mexico and
others, so that you have to adapt the code rather than copy it. Do every task before opening the solution.
If you are stuck, use the AI assistant in VS Code on the task cell, but ask it for a hint rather than the
answer.

Chapters build on one another. Run the setup cell first, then work in order.

0. Setup
1. Python you need
2. Series and DataFrame
3. First look at the corpus
4. Selecting, filtering, sorting
5. Cleaning
6. New columns
7. Keyword counting
8. groupby
9. Reshaping
10. World Bank data and merge
11. Time series
12. Plots
13. Statistics
14. Word frequencies
15. Your project

## Chapter 0: Setup

Run the next cell once per session (and again after every kernel restart). It imports the libraries,
creates the project folders, and defines loaders for the data. You do not need to read the loader code now;
Chapter 3 explains what it does.

**Data.** Put the Kaggle file `un-general-debates.csv` in `data/raw/` when you have it. Without it, the
cell builds a synthetic practice corpus with the same columns and a fake World Bank download. The patterns in
the practice corpus are artificial; swap in the real data as soon as you can.

In [11]:
import os, re, glob, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

for folder in ["data/raw", "data/processed", "figures"]:
    os.makedirs(folder, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 80)


def load_from_txt_files(root):
    """Build the speeches table from the Dataverse folder of .txt files (CODE_SESSION_YEAR.txt)."""
    pattern = re.compile(r"([A-Z]{2,4})_(\d+)_(\d{4})\.txt$")
    rows = []
    for path in glob.glob(os.path.join(root, "**", "*.txt"), recursive=True):
        m = pattern.search(os.path.basename(path))
        if m is None:
            continue
        with open(path, encoding="utf-8", errors="replace") as f:
            text = f.read()
        rows.append({"country": m.group(1), "session": int(m.group(2)),
                     "year": int(m.group(3)), "text": text})
    return pd.DataFrame(rows)


def make_synthetic_corpus(seed=42):
    """Artificial practice corpus with the same four columns plus a fake World Bank download."""
    rng = np.random.default_rng(seed)
    countries = [
        ("CHE", "Europe", 4000), ("DEU", "Europe", 2800), ("FRA", "Europe", 2900), ("GBR", "Europe", 2300),
        ("ITA", "Europe", 2100), ("SWE", "Europe", 4200), ("POL", "Europe", 900), ("GRC", "Europe", 1300),
        ("USA", "Americas", 5200), ("CAN", "Americas", 4100), ("MEX", "Americas", 700), ("BRA", "Americas", 450),
        ("ARG", "Americas", 1300), ("CHL", "Americas", 900), ("COL", "Americas", 350), ("PER", "Americas", 500),
        ("CHN", "Asia", 110), ("IND", "Asia", 110), ("JPN", "Asia", 2000), ("KOR", "Asia", 280),
        ("IDN", "Asia", 80), ("PAK", "Asia", 170), ("THA", "Asia", 190), ("PHL", "Asia", 190),
        ("NGA", "Africa", 220), ("KEN", "Africa", 140), ("EGY", "Africa", 230), ("ZAF", "Africa", 800),
        ("GHA", "Africa", 250), ("ETH", "Africa", 80), ("TZA", "Africa", 100), ("MAR", "Africa", 260),
        ("AUS", "Oceania", 3300), ("NZL", "Oceania", 2400), ("SAU", "Asia", 1000), ("TUR", "Asia", 500),
    ]
    generic = [
        "Mr. President, I congratulate you on your election to the presidency of this session.",
        "My delegation reaffirms its commitment to the principles of the Charter of the United Nations.",
        "The international community must act with unity and resolve.",
        "We welcome the report of the Secretary-General and support his efforts.",
        "Our country remains a firm believer in multilateralism and dialogue.",
        "The reform of the Security Council remains a priority for my Government.",
        "We call on all parties to respect international law and human rights.",
        "Peace and security are the foundation of prosperity for all nations.",
        "I thank the Assembly for its attention.",
        "The situation in the Middle East continues to demand our full attention.",
        "Nuclear disarmament must remain at the top of the agenda of this Organization.",
        "My Government attaches great importance to the work of the specialized agencies.",
    ]
    topical = {
        "trade": ["International trade must be free and fair for all countries.",
                  "Protectionism in the developed world harms the exports of developing nations.",
                  "The multilateral trade system needs to be strengthened.",
                  "Trade barriers and tariffs distort the world economy."],
        "debt": ["The external debt burden is crushing the economies of the developing world.",
                 "We call for debt relief and a restructuring of debt repayments.",
                 "Debt servicing absorbs resources that should go to health and education."],
        "development": ["Economic development is the central challenge of our time.",
                        "Development assistance must be increased to the agreed targets.",
                        "Poverty eradication requires sustained development cooperation."],
        "climate": ["Climate change is a threat to the survival of our planet.",
                    "Greenhouse gas emissions must be reduced substantially.",
                    "We urge all states to implement the climate agreement."],
        "terrorism": ["Terrorism is a threat to all nations and must be defeated.",
                      "We condemn all acts of terrorism wherever they occur."],
    }
    rows, gdp_rows = [], []
    for iso, region, base in countries:
        growth = rng.normal(0.025, 0.008)
        for year in range(1970, 2019):
            t = year - 1970
            gdp = base * (1 + growth) ** t * np.exp(rng.normal(0, 0.05))
            log_gdp = np.log(gdp)
            gdp_rows.append({"Country Name": iso, "Country Code": iso, "region": region,
                             "year": str(year), "gdp": round(gdp * 3.5, 1)})
            p = {
                "trade": 2 + 0.6 * (log_gdp - 6) + 0.04 * t,
                "debt": max(0.3, 6 - 1.2 * (log_gdp - 6) + (3 if 1980 <= year <= 1995 else 0)),
                "development": max(0.5, 8 - 1.5 * (log_gdp - 6)),
                "climate": 0.1 + (0.0 if year < 1988 else 0.25 * (year - 1988)),
                "terrorism": 0.3 + (3.0 if year >= 2001 else 0),
            }
            n_sent = int(rng.integers(50, 110))
            sents = []
            for _ in range(n_sent):
                u = rng.random() * 100
                cum, chosen = 0, None
                for kw, rate in p.items():
                    cum += max(rate, 0)
                    if u < cum:
                        chosen = kw
                        break
                if chosen is None:
                    sents.append(generic[rng.integers(len(generic))])
                else:
                    pool = topical[chosen]
                    sents.append(pool[rng.integers(len(pool))])
            rows.append({"country": iso, "session": year - 1945, "year": year, "text": " ".join(sents)})
    df = pd.DataFrame(rows)
    idx = rng.choice(df.index, 40, replace=False)
    df.loc[idx[:15], "text"] = "\ufeff" + df.loc[idx[:15], "text"]
    df.loc[idx[15:30], "text"] = df.loc[idx[15:30], "text"].str.replace(". ", ".\n\n", regex=False)
    df.loc[idx[30:], "text"] = "   " + df.loc[idx[30:], "text"].str.replace(" ", "   ", regex=False) + "  "
    df = pd.concat([df, df.iloc[[5, 200, 900]]], ignore_index=True)
    df.loc[df.index[-1], "country"] = "chn"
    # fake World Bank download in the real file layout
    g = pd.DataFrame(gdp_rows)
    wide = g.pivot(index=["Country Name", "Country Code"], columns="year", values="gdp").reset_index()
    wide.insert(2, "Indicator Name", "GDP per capita (current US$)")
    wide.insert(3, "Indicator Code", "NY.GDP.PCAP.CD")
    wide[""] = np.nan
    with open("data/raw/worldbank_gdp_pc_wide.csv", "w", encoding="utf-8") as f:
        f.write('"Data Source","World Development Indicators",\n\n"Last Updated Date","2026-01-01",\n\n')
        wide.to_csv(f, index=False)
    latest = g[g["year"] == "2018"].set_index("Country Code")["gdp"]
    income = pd.qcut(latest, 4, labels=["Low income", "Lower middle income",
                                        "Upper middle income", "High income"])
    pd.DataFrame({
        "Country Code": latest.index,
        "Region": [dict((c, r) for c, r, _ in countries)[c] for c in latest.index],
        "IncomeGroup": income.values.astype(str),
        "SpecialNotes": "",
        "TableName": latest.index,
    }).to_csv("data/raw/worldbank_country_metadata.csv", index=False)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


KAGGLE_CSV = "data/raw/un-general-debates.csv"
DATAVERSE_DIR = "data/raw/ungdc"
SYNTHETIC = False
if os.path.exists(KAGGLE_CSV):
    raw = pd.read_csv(KAGGLE_CSV)
    print("Loaded Kaggle CSV:", raw.shape)
elif os.path.isdir(DATAVERSE_DIR):
    raw = load_from_txt_files(DATAVERSE_DIR)
    print("Loaded Dataverse text files:", raw.shape)
else:
    SYNTHETIC = True
    raw = make_synthetic_corpus()
    print("Built synthetic practice corpus:", raw.shape, "(patterns are artificial)")

print("pandas", pd.__version__, "| numpy", np.__version__)

Built synthetic practice corpus: (1767, 4) (patterns are artificial)
pandas 2.3.3 | numpy 2.0.2


**Jupyter in VS Code, the five things you need**

- `Shift+Enter` runs a cell and moves on. The last expression in a cell is displayed; use `print()` for anything else.
- Variables live in the kernel. Restart the kernel (circular arrow at the top) when things get confusing, then run from the top.
- `df.head?` in a cell shows the documentation. Type `df.` and wait for the autocomplete list.
- Add a cell with the `+ Code` button, or `Esc` then `b`.
- Hidden solutions: click the "Solution" line under each task to expand it.

## Chapter 1: The Python you need

Six things: variables, lists, dictionaries, strings, loops, functions. Short examples, then you.

### 1.1 Variables, lists, indexing

A list is ordered. Index from 0. Negative index counts from the end. A slice `a[start:stop]` includes `start`
and excludes `stop`.

In [2]:
codes = ["CHE", "DEU", "FRA", "USA"]
print(codes[0], codes[-1])    # first, last
print(codes[1:3])             # positions 1 and 2
print(len(codes))
codes.append("CHN")
print(codes)

CHE USA
['DEU', 'FRA']
4
['CHE', 'DEU', 'FRA', 'USA', 'CHN']


**Task 1.1**

Make a list `years` with the values 1970, 1980, 1990, 2000, 2010. Print the second element, the last two
elements (as a slice), and the length. Then append 2020 and print the list.

In [10]:
year = [1970, 1980, 1990, 2000, 2010]

print(year[1])
print(year[-1], year[-2:])
print(len(year))
year.append(2030)
print(len(year))
print(year)

1980
2010 [2000, 2010]
5
6
[1970, 1980, 1990, 2000, 2010, 2030]


<details>
<summary>Solution 1.1</summary>

```python
years = [1970, 1980, 1990, 2000, 2010]
print(years[1])
print(years[-2:])
print(len(years))
years.append(2020)
print(years)
```

Output on the practice corpus:

```
1980
[2000, 2010]
5
[1970, 1980, 1990, 2000, 2010, 2020]
```
</details>

### 1.2 Dictionaries

Keys map to values. `d[key]` reads, `d[key] = value` writes, `key in d` tests. `d.get(key, default)` avoids
an error for missing keys.

In [ ]:
gdp = {"CHE": 45000, "DEU": 31000}
gdp["USA"] = 29000
print(gdp["CHE"], "USA" in gdp, "IND" in gdp)
print(gdp.get("IND", "unknown"))
print(list(gdp.keys()))

45000 True False
unknown
['CHE', 'DEU', 'USA']


**Task 1.2**

Build a dictionary `capital` with Brazil (BRA) -> "Brasilia", India (IND) -> "New Delhi". Add Kenya (KEN) ->
"Nairobi". Print the capital of India, test whether "JPN" is a key, and print the value for "JPN" with the
default "not stored" using `.get`.

In [ ]:
# TASK 1.2
capital = {"BRA": "Brasilia", "IND": "New Delhi", "KEN": "Nairobi", "CHE": "Bern"}
capital["USA"] = "Washington"

print(capital["BRA"], "USA" in capital, "GER" in capital)

print(capital.get("JPN", "dont know"))


Brasilia True False
dont know


<details>
<summary>Solution 1.2</summary>

```python
capital = {"BRA": "Brasilia", "IND": "New Delhi"}
capital["KEN"] = "Nairobi"
print(capital["IND"])
print("JPN" in capital)
print(capital.get("JPN", "not stored"))
```

Output on the practice corpus:

```
New Delhi
False
not stored
```
</details>

### 1.3 Strings

Everything here exists later as a pandas `.str` method for whole columns.

In [19]:
s = "  Trade, trade and TRADE.  "
print(s.strip())
print(s.lower().count("trade"))
print(s.split())                 # list of words
print(s.lower().replace("trade", "commerce"))
print(f"{len(s.split())} words")  # f-string

Trade, trade and TRADE.
3
['Trade,', 'trade', 'and', 'TRADE.']
  commerce, commerce and commerce.  
4 words


**Task 1.3**

Given `t = "   Debt relief, debt relief, DEBT RELIEF!  "`: print it without leading and trailing spaces,
count how often "debt" appears ignoring case, print the number of words, and print the text with "relief"
replaced by "forgiveness". Use an f-string for the word count: `"... has N words"`.

In [3]:
# TASK 1.3
t = "   Debt relief, debt relief, DEBT RELIEF!  "

print(t.strip())
print(t.lower().count("debt"))
print(f"{len(t.split())} wordies")
print(t.replace("relief", "forgiveness"))
print(f"The text has {len(t.split())} words in it")

Debt relief, debt relief, DEBT RELIEF!
3
6 wordies
   Debt forgiveness, debt forgiveness, DEBT RELIEF!  
The text has 6 words in it


<details>
<summary>Solution 1.3</summary>

```python
t = "   Debt relief, debt relief, DEBT RELIEF!  "
print(t.strip())
print(t.lower().count("debt"))
print(f"The text has {len(t.split())} words")
print(t.replace("relief", "forgiveness"))
```

Output on the practice corpus:

```
Debt relief, debt relief, DEBT RELIEF!
3
The text has 6 words
   Debt forgiveness, debt forgiveness, DEBT RELIEF!
```
</details>

### 1.4 Loops, conditions, list comprehensions

A list comprehension `[expr for x in items if cond]` builds a list in one line. You will write many of these.

In [2]:
keywords = ["trade", "debt", "war"]
text = "Trade is good. Debt is bad. Trade again."
for kw in keywords:
    n = text.lower().count(kw)
    if n == 0:
        print(kw, "not mentioned")
    else:
        print(kw, "mentioned", n, "times")

lengths = [len(kw) for kw in keywords]
print(lengths)
short = [kw for kw in keywords if len(kw) <= 4]
print(short)

trade mentioned 2 times
debt mentioned 1 times
war not mentioned
[5, 4, 3]
['debt', 'war']


**Task 1.4**

`topics = ["climate", "peace", "security", "poverty"]`. (a) With a for loop, print each topic in upper case
together with its length, in the form `CLIMATE 7`. (b) With a list comprehension, build a list of the
topics that contain the letter "e". (c) With a list comprehension, build a list of the topics' first three
letters.

In [14]:
# TASK 1.4
topics = ["climate", "peace", "security", "poverty"]

for tp in topics:
    print(tp.upper(), len(tp))

contains_e = [tp for tp in topics if "e" in tp]
print(contains_e)

threeletters = [tp[:3].upper() for tp in topics]
print(threeletters)

CLIMATE 7
PEACE 5
SECURITY 8
POVERTY 7
['climate', 'peace', 'security', 'poverty']
['CLI', 'PEA', 'SEC', 'POV']


<details>
<summary>Solution 1.4</summary>

```python
topics = ["climate", "peace", "security", "poverty"]
for tp in topics:
    print(tp.upper(), len(tp))
with_e = [tp for tp in topics if "e" in tp]
print(with_e)
first3 = [tp[:3] for tp in topics]
print(first3)
```

Output on the practice corpus:

```
CLIMATE 7
PEACE 5
SECURITY 8
POVERTY 7
['climate', 'peace', 'security', 'poverty']
['cli', 'pea', 'sec', 'pov']
```
</details>

### 1.5 Functions

Anything used twice goes in a function. Default arguments make a function flexible.

In [16]:
def per_thousand(count, n_words, digits=2):
    return round(1000 * count / n_words, digits)

print(per_thousand(7, 2501))
print(per_thousand(7, 2501, digits=4))



2.8
2.7989


**Task 1.5**

Write a function `share(count, total, percent=True)` that returns `count / total` as a percentage (times 100)
by default, and as a plain fraction when `percent=False`. Round to 1 decimal in both cases. Test it with
`share(12, 48)` and `share(12, 48, percent=False)`.

Then write `count_all(text, keywords)` that returns a dictionary mapping each keyword to its
case-insensitive count in `text`. Test with the text "Peace and security. Peace again." and the keywords
`["peace", "security", "war"]`.

In [3]:
def share(count, total, percent = True, digits=1):
    result = (count / total)
    if percent:
        result *= 100
    result = round(result, percent)
    return f"{result}%" if percent else result

print(share(7, 3))

233.3%


<details>
<summary>Solution 1.5</summary>

```python
def share(count, total, percent=True):
    value = count / total
    if percent:
        value = value * 100
    return round(value, 1)

print(share(12, 48), share(12, 48, percent=False))

def count_all(text, keywords):
    low = text.lower()
    return {kw: low.count(kw) for kw in keywords}

print(count_all("Peace and security. Peace again.", ["peace", "security", "war"]))
```

Output on the practice corpus:

```
25.0 0.2
{'peace': 2, 'security': 1, 'war': 0}
```
</details>

## Chapter 2: Series and DataFrame

A **Series** is one column with an index. A **DataFrame** is a table of Series sharing one index. Operations
apply to every element at once (vectorization): no loops over rows.

In [12]:
gdp = pd.Series([45000, 31000, 29000], index=["CHE", "DEU", "USA"], name="gdp_pc")
print(gdp["DEU"])
print(gdp.mean(), gdp.idxmax())
print(gdp / 1000)
print(gdp > 30000)



31000
35000.0 CHE
CHE    45.0
DEU    31.0
USA    29.0
Name: gdp_pc, dtype: float64
CHE     True
DEU     True
USA    False
Name: gdp_pc, dtype: bool


**Task 2.1**

Create a Series `pop` (population in millions) with index BRA, IND, KEN, JPN and values 215, 1420, 55, 125.
Print the value for Kenya, the total, the label of the largest value, the Series in billions
(divide by 1000), and a boolean Series showing which countries have more than 100 million people.

In [30]:
# TASK 2.1

pop = pd.Series([215, 1420, 55, 125], index=["BRA", "IND", "KEN", "JPN"], name="pop_in_mio")
print(pop["KEN"])
print(pop.sum())
print(pop.max(), pop.idxmax())
print(pop / 1000)
print(pop > 100)

55
1815
1420 IND
BRA    0.215
IND    1.420
KEN    0.055
JPN    0.125
Name: pop_in_mio, dtype: float64
BRA     True
IND     True
KEN    False
JPN     True
Name: pop_in_mio, dtype: bool


<details>
<summary>Solution 2.1</summary>

```python
pop = pd.Series([215, 1420, 55, 125], index=["BRA", "IND", "KEN", "JPN"], name="pop_mio")
print(pop["KEN"])
print(pop.sum())
print(pop.idxmax())
print(pop / 1000)
print(pop > 100)
```

Output on the practice corpus:

```
55
1815
IND
BRA    0.215
IND    1.420
KEN    0.055
JPN    0.125
Name: pop_mio, dtype: float64
BRA     True
IND     True
KEN    False
JPN     True
Name: pop_mio, dtype: bool
```
</details>

### 2.2 Building a DataFrame

From a dictionary of lists. Then the first four things to check: `.shape`, `.dtypes`, `.head()`, `.info()`.

In [ ]:
df = pd.DataFrame({
    "country": ["CHE", "DEU", "USA"],
    "year": [1995, 1995, 1995],
    "gdp_pc": [45000, 31000, 29000],
})
print(df.shape)
print(df.dtypes)
df.head()

country    object
year        int64
gdp_pc      int64
dtype: object


,country,year,gdp_pc
0,CHE,1995,45000
1,DEU,1995,31000
2,USA,1995,29000


**Task 2.2**

Build a DataFrame `ex` with columns `country` (BRA, IND, KEN, JPN), `year` (all 2010), `gdp_pc`
(11000, 1350, 1000, 44500) and `pop_mio` (195, 1230, 42, 128). Print its shape and dtypes and show it.

In [37]:
# TASK 2.2

ex = pd.DataFrame({
    "country": ["BRA", "IND", "KEN", "JPN"],
    "year": [2010, 2010, 2010, 2010],
    "gdp_pc": [11000, 1350, 1000, 44500],
    "pop_mio": [195, 1230, 42, 128],

})

print(ex.shape)
print(ex.dtypes)
ex.head()

(4, 4)
country    object
year        int64
gdp_pc      int64
pop_mio     int64
dtype: object


,country,year,gdp_pc,pop_mio
0,BRA,2010,11000,195
1,IND,2010,1350,1230
2,KEN,2010,1000,42
3,JPN,2010,44500,128


<details>
<summary>Solution 2.2</summary>

```python
ex = pd.DataFrame({
    "country": ["BRA", "IND", "KEN", "JPN"],
    "year": [2010, 2010, 2010, 2010],
    "gdp_pc": [11000, 1350, 1000, 44500],
    "pop_mio": [195, 1230, 42, 128],
})
print(ex.shape)
print(ex.dtypes)
ex
```

Output on the practice corpus:

```
(4, 4)
country      str
year       int64
gdp_pc     int64
pop_mio    int64
dtype: object
  country  year  gdp_pc  pop_mio
0     BRA  2010   11000      195
1     IND  2010    1350     1230
2     KEN  2010    1000       42
3     JPN  2010   44500      128
```
</details>

### 2.3 Columns

One name in brackets gives a Series. A list of names gives a DataFrame. Assigning to a new name creates a column.

In [38]:
print(type(df["gdp_pc"]))
print(type(df[["country", "gdp_pc"]]))
df["log_gdp"] = np.log(df["gdp_pc"])
df["gdp_k"] = df["gdp_pc"] / 1000
df

<class 'pandas.core.series.Series'>
<class 'pandas.core.frame.DataFrame'>


,country,year,gdp_pc,log_gdp,gdp_k
0,CHE,1995,45000,10.714418,45.0
1,DEU,1995,31000,10.341742,31.0
2,USA,1995,29000,10.275051,29.0


**Task 2.3**

On `ex`: add a column `gdp_total_bn` equal to GDP per capita times population in millions, divided by 1000
(so the result is in billions of USD). Add `log_pop` as the natural log of `pop_mio`. Then show only the
columns `country` and `gdp_total_bn`.

In [43]:
# TASK 2.3

ex["gdp_total_bn"] = ex["gdp_pc"] * ex["pop_mio"] / 1000
ex["log_pop"] = np.log(ex["pop_mio"])


ex[["gdp_total_bn", "country"]]

,gdp_total_bn,country
0,2145.0,BRA
1,1660.5,IND
2,42.0,KEN
3,5696.0,JPN


<details>
<summary>Solution 2.3</summary>

```python
ex["gdp_total_bn"] = ex["gdp_pc"] * ex["pop_mio"] / 1000
ex["log_pop"] = np.log(ex["pop_mio"])
ex[["country", "gdp_total_bn"]]
```

Output on the practice corpus:

```
country  gdp_total_bn
0     BRA        2145.0
1     IND        1660.5
2     KEN          42.0
3     JPN        5696.0
```
</details>

### 2.4 The index, and methods return copies

`set_index` makes a column the row labels, `reset_index` turns them back. Most methods return a new object:
assign the result, or it is lost.

In [45]:
dfi = df.set_index("country")
print(dfi.loc["CHE", "gdp_pc"])
df.sort_values("gdp_pc")           # displayed, but df is unchanged
print(df["country"].tolist())
df = df.sort_values("gdp_pc")      # now kept
print(df["country"].tolist())

45000
['CHE', 'DEU', 'USA']
['USA', 'DEU', 'CHE']


**Task 2.4**

Set `country` as the index of `ex` (store as `exi`), look up the population of India with `.loc`, then
sort `exi` by `pop_mio` descending and keep the result in `exi`. Print the index order after sorting.
Finally reset the index and show the first two rows.

In [ ]:
# TASK 2.4

exi = ex.set_index("country")
print(exi.loc["IND"])
exi = exi.sort_values("pop_mio", ascending=False)
exi = exi.reset_index().head(2)

print(exi.head(2))

year            2010.000000
gdp_pc          1350.000000
pop_mio         1230.000000
gdp_total_bn    1660.500000
log_pop            7.114769
Name: IND, dtype: float64
  country  year  gdp_pc  pop_mio  gdp_total_bn   log_pop
0     IND  2010    1350     1230        1660.5  7.114769
1     BRA  2010   11000      195        2145.0  5.273000


<details>
<summary>Solution 2.4</summary>

```python
exi = ex.set_index("country")
print(exi.loc["IND", "pop_mio"])
exi = exi.sort_values("pop_mio", ascending=False)
print(exi.index.tolist())
exi.reset_index().head(2)
```

Output on the practice corpus:

```
1230
['IND', 'BRA', 'JPN', 'KEN']
  country  year  gdp_pc  pop_mio  gdp_total_bn   log_pop
0     IND  2010    1350     1230        1660.5  7.114769
1     BRA  2010   11000      195        2145.0  5.273000
```
</details>

## Chapter 3: First look at the corpus

The setup cell produced `raw`: one row per speech, columns `country`, `session`, `year`, `text`. This is the
"tidy" shape: one row per observation, one column per variable. We copy it to `speeches` and look.

In [59]:
speeches = raw.copy()
speeches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1767 entries, 0 to 1766
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   country  1767 non-null   object
 1   session  1767 non-null   int64 
 2   year     1767 non-null   int64 
 3   text     1767 non-null   object
dtypes: int64(2), object(2)
memory usage: 55.3+ KB


In [60]:
speeches.head(3)

,country,session,year,text
0,MAR,66,2011,Nuclear disarmament must remain at the top of the agenda of this Organizatio...
1,ITA,41,1986,The external debt burden is crushing the economies of the developing world. ...
2,AUS,43,1988,We call on all parties to respect international law and human rights. The in...


In [61]:
print(speeches["country"].nunique(), "countries")
print(speeches["year"].min(), "to", speeches["year"].max())
speeches["year"].value_counts().sort_index().head(5)

37 countries
1970 to 2018


year
1970    36
1971    36
1972    36
1973    36
1974    37
Name: count, dtype: int64

**Task 3.1**

How many speeches are in `speeches`? How many distinct session numbers? Show how many speeches there are
in each of the last five years (`value_counts().sort_index().tail(5)`).

In [ ]:
# TASK 3.1
print(len(speeches))
print(speeches["session"].nunique())
print("There were ", speeches["year"].value_counts().sort_index().tail(5), "in the last 5 years!")

counts = speeches["year"].value_counts().sort_index().tail(5)

print("Number of speeches in the last five years in the data set:")
for year, count in counts.items():
    print(f"In {year}, there were {count} Sessions")



1767
49
There were  year
2014    36
2015    36
2016    36
2017    36
2018    36
Name: count, dtype: int64 in the last 5 years!
Number of speeches in the last five years in the data set:
In 2014, there were 36 Sessions
In 2015, there were 36 Sessions
In 2016, there were 36 Sessions
In 2017, there were 36 Sessions
In 2018, there were 36 Sessions


AttributeError: 'int' object has no attribute 'tail'

In [78]:
country_count = (speeches.groupby("year")["country"]
                 .nunique()
                 .sort_index()
                 .tail(20)
)
print(country_count)

counts = (
    speeches.groupby(["year", "country"])
    .size()
    .reset_index(name="speech_count")

)

year_max = counts.groupby("year")["speech_count"].transform("max")
most_per_year = counts[counts["speech_count"] == year_max].sort_values("year")
print(most_per_year.tail(5))

year
1999    36
2000    36
2001    36
2002    36
2003    36
2004    36
2005    36
2006    36
2007    36
2008    36
2009    36
2010    36
2011    36
2012    36
2013    36
2014    36
2015    36
2016    36
2017    36
2018    36
Name: country, dtype: int64
      year country  speech_count
1741  2018     GBR             1
1742  2018     GHA             1
1743  2018     GRC             1
1745  2018     IND             1
1764  2018     ZAF             1


<details>
<summary>Solution 3.1</summary>

```python
print(len(speeches))
print(speeches["session"].nunique())
speeches["year"].value_counts().sort_index().tail(5)
```

Output on the practice corpus:

```
1767
49
year
2014    36
2015    36
2016    36
2017    36
2018    36
Name: count, dtype: int64
```
</details>

### 3.2 Looking at one text

`repr` shows invisible characters such as `\n` or `\ufeff`. Always look at a few raw texts before cleaning.

In [79]:
one = speeches.loc[speeches["country"] == "CHE", "text"].iloc[0]
print(repr(one[:150]))

'The international community must act with unity and resolve. The international community must act with unity and resolve. I thank the Assembly for its'


**Task 3.2**

Show the first 200 characters, with `repr`, of the first speech by Kenya (KEN) in the table. Then print the
number of characters of that speech with `len`.

In [86]:
# TASK 3.2
two = speeches.loc[speeches["country"] == "KEN", "text"].iloc[0]

print(repr(two[:200]))
print(len(two))

'The international community must act with unity and resolve. We call on all parties to respect international law and human rights. Debt servicing absorbs resources that should go to health and educati'
3760


<details>
<summary>Solution 3.2</summary>

```python
ken = speeches.loc[speeches["country"] == "KEN", "text"].iloc[0]
print(repr(ken[:200]))
print(len(ken))
```

Output on the practice corpus:

```
'The international community must act with unity and resolve. We call on all parties to respect international law and human rights. Debt servicing absorbs resources that should go to health and educati'
3760
```
</details>

### 3.3 Saving and reloading

Never edit the raw file. Save processed versions in `data/processed/`. CSV works everywhere; Parquet is
faster and keeps types.

In [ ]:
speeches.to_csv("data/processed/speeches_raw_copy.csv", index=False)
check = pd.read_csv("data/processed/speeches_raw_copy.csv")
print(check.shape == speeches.shape)

**Task 3.3**

Save only the speeches from 2015 onwards to `data/processed/recent.csv` without the index, read the file
back in as `recent`, and print its shape and the minimum year.

In [ ]:
# TASK 3.3

<details>
<summary>Solution 3.3</summary>

```python
speeches[speeches["year"] >= 2015].to_csv("data/processed/recent.csv", index=False)
recent = pd.read_csv("data/processed/recent.csv")
print(recent.shape, recent["year"].min())
```

Output on the practice corpus:

```
(144, 4) 2015
```
</details>

## Chapter 4: Selecting, filtering, sorting

- `df.iloc[rows, cols]` by position (end excluded).
- `df.loc[rows, cols]` by label (end included).
- `df[mask]` where `mask` is a Series of True/False. This is the one you will use most.

In [ ]:
print(speeches.iloc[0:2, 0:3])          # first two rows, first three columns
speeches.loc[0:1, ["country", "year"]]  # labels 0 and 1, named columns

**Task 4.1**

Show rows 10 to 14 (positions) of the columns in positions 0 and 2 with `iloc`. Then show the rows with
labels 5 to 7 of the columns `year` and `session` with `loc`.

In [ ]:
# TASK 4.1

<details>
<summary>Solution 4.1</summary>

```python
print(speeches.iloc[10:15, [0, 2]])
speeches.loc[5:7, ["year", "session"]]
```

Output on the practice corpus:

```
country  year
10     NZL  2005
11     POL  2018
12     CHN  1993
13     THA  1983
14     KEN  1989
   year  session
5  1986       41
6  2008       63
7  2009       64
```
</details>

### 4.2 Boolean masks

Build the condition, look at it, then put it in brackets. `.sum()` on a boolean Series counts the True values.

In [ ]:
is_che = speeches["country"] == "CHE"
print(is_che.sum())
speeches[is_che][["country", "year"]].head(3)

**Task 4.2**

Build a mask `is_bra` for Brazil (BRA). Print how many speeches match. Show the columns `country`, `year`
and `session` of the first four matching rows.

In [ ]:
# TASK 4.2

<details>
<summary>Solution 4.2</summary>

```python
is_bra = speeches["country"] == "BRA"
print(is_bra.sum())
speeches[is_bra][["country", "year", "session"]].head(4)
```

Output on the practice corpus:

```
49
    country  year  session
16      BRA  1992       47
32      BRA  1982       37
105     BRA  1988       43
246     BRA  1983       38
```
</details>

### 4.3 Combining conditions

`&` and, `|` or, `~` not. Every condition in its own parentheses.

In [ ]:
mask = (speeches["country"] == "DEU") & (speeches["year"] >= 2010)
print(mask.sum())
speeches.loc[mask, ["country", "year"]].sort_values("year").head(3)

**Task 4.3**

(a) How many speeches did India (IND) give before 1980? (b) How many speeches are from either Japan (JPN)
or Mexico (MEX) in the 1990s (1990 to 1999 inclusive)? Use `|` for the country part and `&` for the years.
(c) How many speeches are **not** from the year 2000? Use `~`.

In [ ]:
# TASK 4.3

<details>
<summary>Solution 4.3</summary>

```python
a = (speeches["country"] == "IND") & (speeches["year"] < 1980)
print(a.sum())
b = ((speeches["country"] == "JPN") | (speeches["country"] == "MEX")) & (speeches["year"] >= 1990) & (speeches["year"] <= 1999)
print(b.sum())
c = ~(speeches["year"] == 2000)
print(c.sum())
```

Output on the practice corpus:

```
10
20
1731
```
</details>

### 4.4 isin, between, query

`isin(list)` for membership, `between(a, b)` for ranges (both ends included), `query("...")` for readable
conditions written as a string.

In [ ]:
g7 = ["USA", "CAN", "GBR", "FRA", "DEU", "ITA", "JPN"]
print(speeches["country"].isin(g7).sum())
print(speeches["year"].between(1989, 1991).sum())
speeches.query("country == 'USA' and year > 2015")[["country", "year"]]

**Task 4.4**

Define `brics = ["BRA", "IND", "CHN", "ZAF"]` (RUS is not in the practice corpus). (a) Count the speeches
from BRICS countries. (b) Count the speeches from BRICS countries between 2005 and 2015 using `isin` and
`between`. (c) Rewrite (b) with `query` and show the `country` and `year` columns, first five rows.

In [ ]:
# TASK 4.4

<details>
<summary>Solution 4.4</summary>

```python
brics = ["BRA", "IND", "CHN", "ZAF"]
print(speeches["country"].isin(brics).sum())
m = speeches["country"].isin(brics) & speeches["year"].between(2005, 2015)
print(m.sum())
speeches.query("country in @brics and 2005 <= year <= 2015")[["country", "year"]].head()
```

Output on the practice corpus:

```
196
44
    country  year
48      IND  2011
71      CHN  2006
165     ZAF  2012
197     IND  2006
212     CHN  2005
```
</details>

### 4.5 Sorting and counting

`sort_values` by one or more columns; `ascending` can be a list. `value_counts` counts categories.

In [ ]:
print(speeches.sort_values(["year", "country"]).head(3)[["year", "country"]])
print(speeches.sort_values(["year", "country"], ascending=[False, True]).head(3)[["year", "country"]])
speeches["country"].value_counts().head(3)

**Task 4.5**

(a) Show the three most recent speeches of Kenya (KEN): columns `year`, `session`, sorted by year descending.
(b) Sort the whole table by `country` ascending and `year` descending and show the first three rows'
`country` and `year`. (c) Using `value_counts` on `year`, which year has the fewest speeches?
(Hint: `.idxmin()` on the value_counts result.)

In [ ]:
# TASK 4.5

<details>
<summary>Solution 4.5</summary>

```python
print(speeches[speeches["country"] == "KEN"].sort_values("year", ascending=False)[["year", "session"]].head(3))
print(speeches.sort_values(["country", "year"], ascending=[True, False])[["country", "year"]].head(3))
print(speeches["year"].value_counts().idxmin())
```

Output on the practice corpus:

```
year  session
193   2018       73
1178  2017       72
850   2016       71
     country  year
716      ARG  2018
1305     ARG  2017
678      ARG  2016
2011
```
</details>

### 4.6 Chains

Wrap a sequence of methods in parentheses and put one step per line.

In [ ]:
(
    speeches
    .loc[speeches["year"] >= 2016]
    .sort_values(["year", "country"])
    .loc[:, ["year", "country"]]
    .head(4)
)

**Task 4.6**

Write one chain that: keeps speeches from Mexico (MEX), sorts by year descending, keeps the columns
`year` and `session`, and shows the first five rows.

In [ ]:
# TASK 4.6

<details>
<summary>Solution 4.6</summary>

```python
(
    speeches
    .loc[speeches["country"] == "MEX"]
    .sort_values("year", ascending=False)
    .loc[:, ["year", "session"]]
    .head()
)
```

Output on the practice corpus:

```
year  session
143   2018       73
1357  2017       72
860   2016       71
501   2015       70
255   2014       69
```
</details>

## Chapter 5: Cleaning

Checklist: types, missing values, duplicates, inconsistent categories, text problems. The examples here
perform the actual cleaning of `speeches`; the tasks make you check and practise each step.

### 5.1 Types and missing values

`pd.to_numeric(..., errors="coerce")` turns unconvertible values into missing (`NaN`). `isna()` finds them.

In [ ]:
speeches["year"] = pd.to_numeric(speeches["year"], errors="coerce")
speeches["session"] = pd.to_numeric(speeches["session"], errors="coerce")
print(speeches.dtypes)
print(speeches.isna().sum())

In [ ]:
before = len(speeches)
speeches = speeches.dropna(subset=["text", "country", "year"])
speeches["year"] = speeches["year"].astype(int)
print("dropped", before - len(speeches), "rows")

**Task 5.1**

Practice on a small toy table. Create `toy = pd.DataFrame({"country": ["BRA", "IND", None, "KEN"],
"year": ["2001", "2002", "x", "2004"], "words": [1200, None, 900, 1500]})`. Convert `year` to numeric
with coerce, print how many missing values each column has, then drop rows with a missing `country` or
`year` and print the result.

In [ ]:
# TASK 5.1

<details>
<summary>Solution 5.1</summary>

```python
toy = pd.DataFrame({"country": ["BRA", "IND", None, "KEN"],
                    "year": ["2001", "2002", "x", "2004"],
                    "words": [1200, None, 900, 1500]})
toy["year"] = pd.to_numeric(toy["year"], errors="coerce")
print(toy.isna().sum())
toy = toy.dropna(subset=["country", "year"])
print(toy)
```

Output on the practice corpus:

```
country    1
year       1
words      1
dtype: int64
  country    year   words
0     BRA  2001.0  1200.0
1     IND  2002.0     NaN
3     KEN  2004.0  1500.0
```
</details>

### 5.2 Duplicates

`duplicated(subset=..., keep=False)` flags all rows that share a key. Inspect before dropping. Our rule:
one speech per country-year, keep the longest.

In [ ]:
dup = speeches.duplicated(subset=["country", "year"], keep=False)
print("rows in duplicated country-years:", dup.sum())
speeches[dup].sort_values(["country", "year"])[["country", "year", "session"]]

In [ ]:
speeches["text_len"] = speeches["text"].str.len()
speeches = (
    speeches
    .sort_values("text_len", ascending=False)
    .drop_duplicates(subset=["country", "year"], keep="first")
    .drop(columns="text_len")
)
print(len(speeches))

**Task 5.2**

`toy2 = pd.DataFrame({"country": ["BRA", "BRA", "IND", "IND", "KEN"], "year": [2001, 2001, 2002, 2002, 2003],
"words": [1200, 1500, 900, 800, 1100]})`. Flag the rows that share a country-year, then keep, for each
country-year, the row with the most words. Print the result (it should have 3 rows).

In [ ]:
# TASK 5.2

<details>
<summary>Solution 5.2</summary>

```python
toy2 = pd.DataFrame({"country": ["BRA", "BRA", "IND", "IND", "KEN"],
                     "year": [2001, 2001, 2002, 2002, 2003],
                     "words": [1200, 1500, 900, 800, 1100]})
print(toy2.duplicated(subset=["country", "year"], keep=False))
toy2 = toy2.sort_values("words", ascending=False).drop_duplicates(subset=["country", "year"])
print(toy2.sort_values("country"))
```

Output on the practice corpus:

```
0     True
1     True
2     True
3     True
4    False
dtype: bool
  country  year  words
1     BRA  2001   1500
2     IND  2002    900
4     KEN  2003   1100
```
</details>

### 5.3 Inconsistent categories and the `.str` accessor

`.str` applies a string method to every element of a column. Chain several.

In [ ]:
print(speeches["country"].str.isupper().all())
speeches["country"] = speeches["country"].str.strip().str.upper()
print(speeches["country"].str.isupper().all())
# upper-casing can create new duplicates, so resolve them again
speeches["text_len"] = speeches["text"].str.len()
speeches = (speeches.sort_values("text_len", ascending=False)
                    .drop_duplicates(subset=["country", "year"]).drop(columns="text_len"))
print(len(speeches))

**Task 5.3**

`names = pd.Series([" brazil", "INDIA ", "Kenya", "  japan  "])`. Produce a cleaned Series with no
surrounding spaces and the first letter capitalised (`.str.strip()`, `.str.capitalize()`). Then produce
a Series of three-letter upper-case abbreviations from the first three letters (`.str[:3]`, `.str.upper()`).

In [ ]:
# TASK 5.3
names = pd.Series([" brazil", "INDIA ", "Kenya", "  japan  "])

<details>
<summary>Solution 5.3</summary>

```python
names = pd.Series([" brazil", "INDIA ", "Kenya", "  japan  "])
clean = names.str.strip().str.capitalize()
print(clean)
print(clean.str[:3].str.upper())
```

Output on the practice corpus:

```
0    Brazil
1     India
2     Kenya
3     Japan
dtype: str
0    BRA
1    IND
2    KEN
3    JAP
dtype: str
```
</details>

### 5.4 Text problems

Count the problems first (`.str.contains`), then fix them in one chain. `regex=True` with `\s+` collapses
any run of whitespace, including newlines, into one space.

In [ ]:
print("BOM:", speeches["text"].str.contains("\ufeff").sum(),
      "| newlines:", speeches["text"].str.contains("\n").sum(),
      "| double spaces:", speeches["text"].str.contains("  ").sum())
speeches["text"] = (
    speeches["text"]
    .str.replace("\ufeff", "", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
print("BOM:", speeches["text"].str.contains("\ufeff").sum(),
      "| newlines:", speeches["text"].str.contains("\n").sum(),
      "| double spaces:", speeches["text"].str.contains("  ").sum())

**Task 5.4**

`messy = pd.Series(["\ufeffWe  call for\n\npeace.  ", "  Trade   must be\tfree."])`. Count how many entries
contain a tab (`\t`). Clean the Series the same way as above and print each cleaned entry with `repr`
(loop over the Series).

In [ ]:
# TASK 5.4
messy = pd.Series(["\ufeffWe  call for\n\npeace.  ", "  Trade   must be\tfree."])

<details>
<summary>Solution 5.4</summary>

```python
messy = pd.Series(["\ufeffWe  call for\n\npeace.  ", "  Trade   must be\tfree."])
print(messy.str.contains("\t").sum())
cleaned = messy.str.replace("\ufeff", "", regex=False).str.replace(r"\s+", " ", regex=True).str.strip()
for t in cleaned:
    print(repr(t))
```

Output on the practice corpus:

```
1
'We call for peace.'
'Trade must be free.'
```
</details>

### 5.5 Regular expressions in ten symbols

| pattern | matches |
|---|---|
| `\s` `\d` `\w` | whitespace, digit, word character |
| `\b` | word boundary |
| `.` | any character |
| `+` `*` `?` | one or more, zero or more, zero or one |
| `[abc]` | one of a, b, c |
| `(x|y)` | x or y |
| `^` `$` | start, end |

`\b` matters: `trade` also matches "traded"; `\btrade\b` matches only the whole word.

In [ ]:
t = "Trade talks: traders traded trademarks. Free trade!"
print(len(re.findall("trade", t.lower())))
print(len(re.findall(r"\btrade\b", t.lower())))
print(len(re.findall(r"\btrad(e|ed|ers?|ing)\b", t.lower())))

**Task 5.5**

`u = "Debt, indebted nations and debtors. Debt relief, not debts!"`. Count, ignoring case: (a) all occurrences
of the letters "debt" anywhere, (b) only the whole word "debt", (c) the whole words debt, debts or debtors
with one pattern using a group.

In [ ]:
# TASK 5.5
u = "Debt, indebted nations and debtors. Debt relief, not debts!"

<details>
<summary>Solution 5.5</summary>

```python
u = "Debt, indebted nations and debtors. Debt relief, not debts!"
print(len(re.findall("debt", u.lower())))
print(len(re.findall(r"\bdebt\b", u.lower())))
print(len(re.findall(r"\bdebt(s|ors)?\b", u.lower())))
```

Output on the practice corpus:

```
5
2
4
```
</details>

### 5.6 Finish: lowercase copy, order, save

In [ ]:
speeches["text_lower"] = speeches["text"].str.lower()
speeches = speeches.sort_values(["year", "country"]).reset_index(drop=True)
speeches.to_csv("data/processed/speeches_clean.csv", index=False)
print(speeches.shape)
speeches.head(2)

**Task 5.6**

Three checks on the cleaned table. (a) Is every `country` value exactly three characters long?
(`.str.len() == 3`, then `.all()`). (b) How many speeches start with "Mr. President"? (`.str.startswith`).
(c) What share of speeches contain the phrase "international community" (use `text_lower`)?

In [ ]:
# TASK 5.6

<details>
<summary>Solution 5.6</summary>

```python
print((speeches["country"].str.len() == 3).all())
print(speeches["text"].str.startswith("Mr. President").sum())
print(speeches["text_lower"].str.contains("international community").mean())
```

Output on the practice corpus:

```
True
120
0.9954648526077098
```
</details>

## Chapter 6: New columns

Fastest to slowest: vectorized arithmetic and `.str`, `np.where` and `pd.cut`, `.map` with a dictionary,
`.apply` with a function. Try them in that order.

In [ ]:
speeches["n_words"] = speeches["text"].str.split().str.len()
speeches["n_sentences"] = speeches["text"].str.count(r"[.!?]")
speeches["words_per_sentence"] = speeches["n_words"] / speeches["n_sentences"]
speeches[["n_words", "n_sentences", "words_per_sentence"]].describe().round(1)

**Task 6.1**

Add `n_characters` (length of `text`), `avg_word_len` (characters divided by words), and `n_commas`
(count of ","). Show `describe()` of the three, rounded to 2 decimals. Which speech (country, year) has the
highest `avg_word_len`? (Hint: `.loc[speeches["avg_word_len"].idxmax(), ["country", "year"]]`.)

In [ ]:
# TASK 6.1

<details>
<summary>Solution 6.1</summary>

```python
speeches["n_characters"] = speeches["text"].str.len()
speeches["avg_word_len"] = speeches["n_characters"] / speeches["n_words"]
speeches["n_commas"] = speeches["text"].str.count(",")
print(speeches[["n_characters", "avg_word_len", "n_commas"]].describe().round(2))
print(speeches.loc[speeches["avg_word_len"].idxmax(), ["country", "year"]])
```

Output on the practice corpus:

```
n_characters  avg_word_len  n_commas
count       1764.00       1764.00   1764.00
mean        5567.06          6.36      5.34
std         1242.27          0.06      2.57
min         3367.00          6.14      0.00
25%         4474.00          6.32      3.00
50%         5548.50          6.36      5.00
75%         6674.00          6.40      7.00
max         7995.00          6.68     15.00
country     TZA
year       1995
Name: 933, dtype: object
```
</details>

### 6.2 Conditional and binned columns

`np.where(cond, a, b)`. `pd.cut(values, bins, labels)` for intervals. Integer division `// 10 * 10` for decades.

In [ ]:
speeches["post_cold_war"] = np.where(speeches["year"] >= 1991, 1, 0)
speeches["decade"] = (speeches["year"] // 10 * 10).astype(str) + "s"
print(speeches["post_cold_war"].value_counts())
print(speeches["decade"].value_counts().sort_index())

**Task 6.2**

(a) Add `long_speech`, 1 if `n_words` is above the median of `n_words`, else 0, and print its mean (the share
of long speeches). (b) Add `era` with `pd.cut`: "cold_war" for years up to 1990, "unipolar" for 1991 to 2007,
"recent" from 2008. Print the value counts. (c) Add `half_decade` as the year rounded down to a multiple
of 5 (e.g. 1987 -> 1985). Print the unique values sorted.

In [ ]:
# TASK 6.2

<details>
<summary>Solution 6.2</summary>

```python
speeches["long_speech"] = np.where(speeches["n_words"] > speeches["n_words"].median(), 1, 0)
print(speeches["long_speech"].mean())
speeches["era"] = pd.cut(speeches["year"], bins=[1900, 1990, 2007, 2100], labels=["cold_war", "unipolar", "recent"])
print(speeches["era"].value_counts())
speeches["half_decade"] = speeches["year"] // 5 * 5
print(sorted(speeches["half_decade"].unique()))
```

Output on the practice corpus:

```
0.5
era
cold_war    756
unipolar    612
recent      396
Name: count, dtype: int64
[np.int64(1970), np.int64(1975), np.int64(1980), np.int64(1985), np.int64(1990), np.int64(1995), np.int64(2000), np.int64(2005), np.int64(2010), np.int64(2015)]
```
</details>

### 6.3 map and apply

`.map(dict)` recodes; unmatched values become NaN, so check. `.apply(func)` runs a Python function per element.

In [ ]:
region_map = {"CHE": "Europe", "DEU": "Europe", "USA": "Americas"}
tmp = speeches["country"].map(region_map)
print(tmp.value_counts(dropna=False).head(3))

def first_sentence(text):
    return text.split(". ")[0]

speeches["opening"] = speeches["text"].apply(first_sentence)
speeches["opening"].value_counts().head(3)

**Task 6.3**

(a) Build `lang_map` mapping BRA -> "pt", IND -> "en", KEN -> "en", JPN -> "ja", MEX -> "es" and create a
column `lang` with `.map`. How many speeches have a missing `lang`? (b) Write a function `last_word(text)`
that returns the last word of a text without punctuation (hint: `text.split()[-1].strip(".!?")`) and apply
it to create `closing_word`. Show its five most common values.

In [ ]:
# TASK 6.3

<details>
<summary>Solution 6.3</summary>

```python
lang_map = {"BRA": "pt", "IND": "en", "KEN": "en", "JPN": "ja", "MEX": "es"}
speeches["lang"] = speeches["country"].map(lang_map)
print(speeches["lang"].isna().sum())

def last_word(text):
    return text.split()[-1].strip(".!?")

speeches["closing_word"] = speeches["text"].apply(last_word)
print(speeches["closing_word"].value_counts().head())
```

Output on the practice corpus:

```
1519
closing_word
attention     234
nations       139
Government    136
agencies      124
rights        122
Name: count, dtype: int64
```
</details>

### 6.4 assign inside chains, housekeeping

`assign(new=lambda d: ...)` creates a column mid-chain; `d` is the DataFrame at that point. `rename`, `drop`,
`select_dtypes` manage columns.

In [ ]:
(speeches
 .assign(log_words=lambda d: np.log(d["n_words"]))
 .loc[:, ["country", "year", "log_words"]]
 .head(2))

In [ ]:
speeches = speeches.drop(columns=["opening", "lang", "closing_word"], errors="ignore")
print(speeches.select_dtypes(include="number").columns.tolist())

**Task 6.4**

In one chain: filter to Japan (JPN), add a column `words_k` (words in thousands, with `assign` and a
lambda), keep `year` and `words_k`, sort by `words_k` descending, show the top three. Then drop the column
`half_decade` from `speeches` and print the remaining column names.

In [ ]:
# TASK 6.4

<details>
<summary>Solution 6.4</summary>

```python
print(speeches
      .loc[speeches["country"] == "JPN"]
      .assign(words_k=lambda d: d["n_words"] / 1000)
      .loc[:, ["year", "words_k"]]
      .sort_values("words_k", ascending=False)
      .head(3))
speeches = speeches.drop(columns=["half_decade"])
print(speeches.columns.tolist())
```

Output on the practice corpus:

```
year  words_k
882  1994    1.215
54   1971    1.209
378  1980    1.207
['country', 'session', 'year', 'text', 'text_lower', 'n_words', 'n_sentences', 'words_per_sentence', 'n_characters', 'avg_word_len', 'n_commas', 'post_cold_war', 'decade', 'long_speech', 'era']
```
</details>

### Checkpoint 6

Later chapters use `era` and `long_speech` from Task 6.2. This cell creates them if you skipped the task,
and does nothing otherwise. Each chapter from here on ends with such a checkpoint.

In [ ]:
if "era" not in speeches.columns:
    speeches["era"] = pd.cut(speeches["year"], bins=[1900, 1990, 2007, 2100], labels=["cold_war", "unipolar", "recent"])
if "long_speech" not in speeches.columns:
    speeches["long_speech"] = np.where(speeches["n_words"] > speeches["n_words"].median(), 1, 0)
print(speeches.columns.tolist())

## Chapter 7: Keyword counting

The core text measure: count words from a list, normalize per 1,000 words. A topic is a regex with
alternatives in a group. Naming convention: `kw_<topic>` for raw counts, `kw_<topic>_pt` per thousand,
`mentions_<topic>` for 0/1.

In [ ]:
keywords = {
    "trade": r"\b(trade|trading|tariffs?|exports?|imports?|protectionism)\b",
    "debt":  r"\b(debts?|indebted|creditors?|debt relief)\b",
}
for topic, pattern in keywords.items():
    speeches[f"kw_{topic}"] = speeches["text_lower"].str.count(pattern)
    speeches[f"kw_{topic}_pt"] = 1000 * speeches[f"kw_{topic}"] / speeches["n_words"]
    speeches[f"mentions_{topic}"] = (speeches[f"kw_{topic}"] > 0).astype(int)
speeches.filter(like="kw_").describe().round(2)

**Task 7.1**

Add four more topics to the `keywords` dictionary and compute all three columns for each, by adding to the
dictionary and running the same loop (write the loop yourself):
- development: development, developing, poverty, underdevelop(any ending)
- climate: climate, global warming, greenhouse, emission(s)
- terrorism: terrorism, terrorist(s)
- peace: peace, peaceful, ceasefire

Then show `describe()` of all `_pt` columns, rounded to 2.

In [ ]:
# TASK 7.1

<details>
<summary>Solution 7.1</summary>

```python
keywords["development"] = r"\b(development|developing|poverty|underdevelop\w*)\b"
keywords["climate"] = r"\b(climate|global warming|greenhouse|emissions?)\b"
keywords["terrorism"] = r"\b(terrorism|terrorists?)\b"
keywords["peace"] = r"\b(peace|peaceful|ceasefire)\b"
for topic, pattern in keywords.items():
    speeches[f"kw_{topic}"] = speeches["text_lower"].str.count(pattern)
    speeches[f"kw_{topic}_pt"] = 1000 * speeches[f"kw_{topic}"] / speeches["n_words"]
    speeches[f"mentions_{topic}"] = (speeches[f"kw_{topic}"] > 0).astype(int)
speeches.filter(like="_pt").describe().round(2)
```

Output on the practice corpus:

```
kw_trade_pt  kw_debt_pt  kw_development_pt  kw_climate_pt  kw_terrorism_pt  kw_peace_pt
count      1764.00     1764.00            1764.00        1764.00          1764.00      1764.00
mean          4.86        6.79              10.31           2.95             1.28         6.06
std           3.55        4.39               4.97           3.82             1.84         2.60
min           0.00        0.00               0.00           0.00             0.00         0.00
25%           2.25        3.42               6.72           0.00             0.00         4.26
50%           4.31        6.31               9.93           1.24             0.00         5.90
75%           6.99        9.41              13.39           5.16             2.00         7.77
max          27.31       23.36              31.91          21.07            10.83        16.93
```
</details>

### 7.2 Finding and checking extreme cases

`nlargest(n, col)` is sort plus head. Always look at the matches in context before trusting a dictionary.

In [ ]:
print(speeches.nlargest(3, "kw_trade_pt")[["country", "year", "kw_trade", "kw_trade_pt"]])

def show_matches(text, pattern, n=3, width=50):
    for i, m in enumerate(re.finditer(pattern, text)):
        if i >= n:
            break
        print("...", text[max(0, m.start()-width):m.end()+width], "...")

top = speeches.nlargest(1, "kw_trade_pt").iloc[0]
show_matches(top["text_lower"], keywords["trade"])

**Task 7.2**

(a) Show the five speeches with the highest climate intensity: `country`, `year`, `kw_climate_pt`.
(b) Show the three speeches with the **lowest** debt intensity among those that mention debt at all
(filter `mentions_debt == 1`, then `nsmallest`). (c) Use `show_matches` to print two matches of the
development pattern in the speech with the highest `kw_development_pt`.

In [ ]:
# TASK 7.2

<details>
<summary>Solution 7.2</summary>

```python
print(speeches.nlargest(5, "kw_climate_pt")[["country", "year", "kw_climate_pt"]])
print(speeches[speeches["mentions_debt"] == 1].nsmallest(3, "kw_debt_pt")[["country", "year", "kw_debt_pt"]])
top_dev = speeches.nlargest(1, "kw_development_pt").iloc[0]
show_matches(top_dev["text_lower"], keywords["development"], n=2)
```

Output on the practice corpus:

```
country  year  kw_climate_pt
1737     EGY  2018      21.067416
1757     SAU  2018      19.362187
1667     FRA  2016      19.316493
1651     THA  2015      18.134715
1746     JPN  2018      17.520216
     country  year  kw_debt_pt
1632     GBR  2015    0.828500
316      POL  1978    0.835422
1470     SWE  2010    0.838223
... of the secretary-general and support his efforts. poverty eradication requires sustained development cooper ...
... s efforts. poverty eradication requires sustained development cooperation. the external debt burden is crushing ...
```
</details>

### 7.3 Shares and combinations

The mean of a 0/1 column is a share. Combine masks for co-occurrence.

In [ ]:
print(speeches.filter(like="mentions_").mean().round(3))
both = (speeches["mentions_trade"] == 1) & (speeches["mentions_debt"] == 1)
print(both.mean())

**Task 7.3**

(a) What share of speeches mention climate but **not** terrorism? (b) What share of Brazilian (BRA) speeches
mention debt? (c) Among speeches after 2000, what share mention both peace and terrorism?

In [ ]:
# TASK 7.3

<details>
<summary>Solution 7.3</summary>

```python
print(((speeches["mentions_climate"] == 1) & (speeches["mentions_terrorism"] == 0)).mean())
print(speeches.loc[speeches["country"] == "BRA", "mentions_debt"].mean())
after = speeches[speeches["year"] > 2000]
print(((after["mentions_peace"] == 1) & (after["mentions_terrorism"] == 1)).mean())
```

Output on the practice corpus:

```
0.17687074829931973
1.0
0.9212962962962963
```
</details>

### Checkpoint 7

Everything after this needs all six topics. This cell fills in any that are missing.

In [ ]:
defaults = {
    "trade": r"\b(trade|trading|tariffs?|exports?|imports?|protectionism)\b",
    "debt": r"\b(debts?|indebted|creditors?|debt relief)\b",
    "development": r"\b(development|developing|poverty|underdevelop\w*)\b",
    "climate": r"\b(climate|global warming|greenhouse|emissions?)\b",
    "terrorism": r"\b(terrorism|terrorists?)\b",
    "peace": r"\b(peace|peaceful|ceasefire)\b",
}
for topic, pattern in defaults.items():
    keywords.setdefault(topic, pattern)
    if f"kw_{topic}_pt" not in speeches.columns:
        speeches[f"kw_{topic}"] = speeches["text_lower"].str.count(keywords[topic])
        speeches[f"kw_{topic}_pt"] = 1000 * speeches[f"kw_{topic}"] / speeches["n_words"]
        speeches[f"mentions_{topic}"] = (speeches[f"kw_{topic}"] > 0).astype(int)
print(sorted(keywords))

## Chapter 8: groupby

```
df.groupby(keys)[columns].aggregation()
```
Split by keys, aggregate per group, combine. The result's index is the group key(s).

In [ ]:
print(speeches.groupby("decade").size())
print(speeches.groupby("year")["kw_trade_pt"].mean().head(3))
speeches.groupby("decade")["kw_trade_pt"].agg(["mean", "median", "count"]).round(2)

**Task 8.1**

(a) Average `kw_debt_pt` per decade (round 2). (b) For each `era`, the mean, min and max of `n_words`
(use `observed=True` in the groupby if you get a warning). (c) Number of speeches per country; show the
five countries with the fewest.

In [ ]:
# TASK 8.1

<details>
<summary>Solution 8.1</summary>

```python
print(speeches.groupby("decade")["kw_debt_pt"].mean().round(2))
print(speeches.groupby("era", observed=True)["n_words"].agg(["mean", "min", "max"]).round(0))
print(speeches.groupby("country").size().sort_values().head())
```

Output on the practice corpus:

```
decade
1970s    6.39
1980s    9.53
1990s    7.44
2000s    5.19
2010s    5.27
Name: kw_debt_pt, dtype: float64
           mean  min   max
era                       
cold_war  883.0  525  1265
unipolar  875.0  514  1227
recent    861.0  531  1232
country
ARG    49
AUS    49
BRA    49
CAN    49
CHE    49
dtype: int64
```
</details>

### 8.2 Named aggregations and several keys

`agg(name=(column, function))` gives you clean output names. Two keys produce a two-level index;
`as_index=False` keeps keys as columns.

In [ ]:
summary = speeches.groupby("country").agg(
    n=("year", "size"),
    first=("year", "min"),
    trade_pt=("kw_trade_pt", "mean"),
)
print(summary.sort_values("trade_pt", ascending=False).head(3).round(2))
speeches.groupby(["decade", "mentions_climate"], as_index=False)["n_words"].mean().head(4)

**Task 8.2**

(a) Build `by_country` with named aggregations: `n_speeches` (size), `last_year` (max of year),
`avg_words` (mean of n_words), `climate_pt` (mean of kw_climate_pt), `debt_pt` (mean of kw_debt_pt).
Show the five countries with the highest `climate_pt`, rounded to 2. (b) Group by `decade` and
`post_cold_war` with `as_index=False` and compute the mean `kw_terrorism_pt`.

In [ ]:
# TASK 8.2

<details>
<summary>Solution 8.2</summary>

```python
by_country = speeches.groupby("country").agg(
    n_speeches=("year", "size"),
    last_year=("year", "max"),
    avg_words=("n_words", "mean"),
    climate_pt=("kw_climate_pt", "mean"),
    debt_pt=("kw_debt_pt", "mean"),
)
print(by_country.sort_values("climate_pt", ascending=False).head().round(2))
print(speeches.groupby(["decade", "post_cold_war"], as_index=False)["kw_terrorism_pt"].mean().round(2))
```

Output on the practice corpus:

```
n_speeches  last_year  avg_words  climate_pt  debt_pt
country                                                       
GHA              49       2018     894.04        3.84     9.10
JPN              49       2018     862.69        3.75     5.75
SWE              49       2018     923.82        3.62     4.22
TZA              49       2018     875.84        3.44    10.20
PHL              49       2018     896.84        3.36     8.47
  decade  post_cold_war  kw_terrorism_pt
0  1970s              0             0.27
1  1980s              0             0.26
2  1990s              0             0.22
3  1990s              1             0.23
4  2000s              1             2.83
5  2010s              1             3.01
```
</details>

### 8.3 transform

`agg` collapses groups; `transform` returns the group statistic for every row. Use it to demean within
groups (that is what fixed effects do) or to compute shares within groups.

In [ ]:
speeches["trade_country_mean"] = speeches.groupby("country")["kw_trade_pt"].transform("mean")
speeches["trade_demeaned"] = speeches["kw_trade_pt"] - speeches["trade_country_mean"]
speeches[["country", "year", "kw_trade_pt", "trade_country_mean", "trade_demeaned"]].head(3).round(2)

**Task 8.3**

(a) Create `debt_year_mean`, the mean `kw_debt_pt` of all speeches in the same year, and `debt_vs_year`,
the speech's `kw_debt_pt` minus that mean. Which country-year has the highest `debt_vs_year`? (b) Create
`words_share_in_year`: the speech's `n_words` divided by the **sum** of `n_words` in that year. Check that
the shares within one year (say 1995) add up to 1.

In [ ]:
# TASK 8.3

<details>
<summary>Solution 8.3</summary>

```python
speeches["debt_year_mean"] = speeches.groupby("year")["kw_debt_pt"].transform("mean")
speeches["debt_vs_year"] = speeches["kw_debt_pt"] - speeches["debt_year_mean"]
print(speeches.nlargest(1, "debt_vs_year")[["country", "year", "debt_vs_year"]])
speeches["words_share_in_year"] = speeches["n_words"] / speeches.groupby("year")["n_words"].transform("sum")
print(speeches.loc[speeches["year"] == 1995, "words_share_in_year"].sum())
```

Output on the practice corpus:

```
country  year  debt_vs_year
1203     IDN  2003     15.704262
0.9999999999999999
```
</details>

### 8.4 pivot_table, crosstab, iterating over groups

In [ ]:
print(pd.pivot_table(speeches, values="kw_climate_pt", index="decade", columns="post_cold_war", aggfunc="mean").round(2))
print(pd.crosstab(speeches["decade"], speeches["mentions_terrorism"], normalize="index").round(2))
for dec, grp in speeches.groupby("decade"):
    print(dec, len(grp), grp["country"].nunique())

**Task 8.4**

(a) Pivot table: rows `era`, columns `decade`, values mean `kw_debt_pt` (add `observed=True`).
(b) Crosstab of `long_speech` against `mentions_climate`, normalized by row. (c) Loop over the groups of
`era` and print, for each, the era name and the country with the highest mean `kw_peace_pt`
(hint: inside the loop, `grp.groupby("country")["kw_peace_pt"].mean().idxmax()`).

In [ ]:
# TASK 8.4

<details>
<summary>Solution 8.4</summary>

```python
print(pd.pivot_table(speeches, values="kw_debt_pt", index="era", columns="decade", aggfunc="mean", observed=True).round(2))
print(pd.crosstab(speeches["long_speech"], speeches["mentions_climate"], normalize="index").round(2))
for era, grp in speeches.groupby("era", observed=True):
    print(era, grp.groupby("country")["kw_peace_pt"].mean().idxmax())
```

Output on the practice corpus:

```
decade    1970s  1980s  1990s  2000s  2010s
era                                        
cold_war   6.39   9.53   9.42    NaN    NaN
unipolar    NaN    NaN   7.22   5.38    NaN
recent      NaN    NaN    NaN   4.41   5.27
mentions_climate     0     1
long_speech                 
0                 0.49  0.51
1                 0.42  0.58
cold_war GHA
unipolar COL
recent SAU
```
</details>

## Chapter 9: Reshaping

Wide: one column per variable or period. Long: one row per unit-variable pair. `melt` goes wide to long,
`pivot` goes long to wide, `unstack` moves an index level to columns.

In [ ]:
pt_cols = [c for c in speeches.columns if c.endswith("_pt") and c.startswith("kw_")]
long = speeches.melt(id_vars=["country", "year"], value_vars=pt_cols, var_name="topic", value_name="per_thousand")
long["topic"] = long["topic"].str.replace("kw_", "").str.replace("_pt", "")
print(long.head(3))
print(long.groupby("topic")["per_thousand"].mean().round(2))

**Task 9.1**

Melt the `mentions_*` columns into a long table `long_m` with id columns `country`, `year`, `decade`, a
`topic` column (strip the "mentions_" prefix) and a `mentions` column. Then compute the share of speeches
mentioning each topic per decade as a groupby on `["decade", "topic"]`, and `unstack("topic")` the result.
Round to 2.

In [ ]:
# TASK 9.1

<details>
<summary>Solution 9.1</summary>

```python
m_cols = [c for c in speeches.columns if c.startswith("mentions_")]
long_m = speeches.melt(id_vars=["country", "year", "decade"], value_vars=m_cols, var_name="topic", value_name="mentions")
long_m["topic"] = long_m["topic"].str.replace("mentions_", "")
print(long_m.groupby(["decade", "topic"])["mentions"].mean().unstack("topic").round(2))
```

Output on the practice corpus:

```
topic   climate  debt  development  peace  terrorism  trade
decade                                                     
1970s      0.05  0.95         0.99   1.00       0.20   0.84
1980s      0.08  0.99         1.00   0.99       0.21   0.86
1990s      0.72  0.96         1.00   0.99       0.19   0.92
2000s      0.95  0.93         0.99   0.99       0.86   0.93
2010s      0.98  0.91         0.99   0.98       0.93   0.95
```
</details>

In [ ]:
wide_again = long.pivot(index=["country", "year"], columns="topic", values="per_thousand")
print(wide_again.head(2).round(2))
wb_like = pd.DataFrame({"Country Code": ["CHE", "DEU"], "1970": [3700, 2800], "1971": [4100, 3000]})
wb_like.melt(id_vars="Country Code", var_name="year", value_name="gdp_pc")

**Task 9.2**

(a) From `long`, build a table with years as rows, topics as columns and the mean `per_thousand` as values
(`pivot_table`), and show the last three rows rounded to 2. (b) `wb2 = pd.DataFrame({"Country Code":
["BRA", "IND", "KEN"], "2000": [3700, 440, 400], "2010": [11000, 1350, 1000], "2020": [6900, 1900, 1900]})`.
Melt it to long with columns `country`, `year`, `gdp_pc`, convert `year` to int, and sort by country and year.

In [ ]:
# TASK 9.2

<details>
<summary>Solution 9.2</summary>

```python
print(long.pivot_table(index="year", columns="topic", values="per_thousand", aggfunc="mean").tail(3).round(2))
wb2 = pd.DataFrame({"Country Code": ["BRA", "IND", "KEN"], "2000": [3700, 440, 400],
                    "2010": [11000, 1350, 1000], "2020": [6900, 1900, 1900]})
wb2_long = (wb2.melt(id_vars="Country Code", var_name="year", value_name="gdp_pc")
               .rename(columns={"Country Code": "country"}))
wb2_long["year"] = wb2_long["year"].astype(int)
print(wb2_long.sort_values(["country", "year"]))
```

Output on the practice corpus:

```
topic  climate  debt  development  peace  terrorism  trade
year                                                      
2016      8.71  4.67         9.22   5.96       2.81   7.05
2017      9.15  5.12         9.64   5.39       2.54   6.88
2018      9.57  5.04         7.67   5.60       3.51   6.15
  country  year  gdp_pc
0     BRA  2000    3700
3     BRA  2010   11000
6     BRA  2020    6900
1     IND  2000     440
4     IND  2010    1350
7     IND  2020    1900
2     KEN  2000     400
5     KEN  2010    1000
8     KEN  2020    1900
```
</details>

## Chapter 10: World Bank data and merge

We attach GDP per capita (`NY.GDP.PCAP.CD`) and the income group to every speech. Target tables:
`gdp` with `country, year, gdp_pc` and `economies` with `country, country_name, wb_region, income_group`.

**Real data, option A:** `pip install wbgapi`, then run the cell below. It is skipped while the practice
corpus is in use.
**Real data, option B:** download the CSV from data.worldbank.org (indicator "GDP per capita (current US$)",
Download, CSV); put the data file as `data/raw/worldbank_gdp_pc_wide.csv` and `Metadata_Country_*.csv`
as `data/raw/worldbank_country_metadata.csv`. The practice setup already wrote files in this layout.

In [ ]:
gdp = economies = None
try:
    if SYNTHETIC:
        raise RuntimeError("practice corpus: using the fake CSV instead")
    import wbgapi as wb
    raw_wb = wb.data.DataFrame("NY.GDP.PCAP.CD", time=range(1960, 2025), labels=False)
    gdp = (raw_wb.reset_index().rename(columns={"economy": "country"})
                 .melt(id_vars="country", var_name="year", value_name="gdp_pc")
                 .assign(year=lambda d: d["year"].str.replace("YR", "").astype(int))
                 .dropna(subset=["gdp_pc"]))
    econ_raw = wb.economy.DataFrame(labels=True)   # if this fails, print(econ_raw.columns) and adjust
    economies = (econ_raw[~econ_raw["aggregate"]].reset_index()
                 .rename(columns={"id": "country", "name": "country_name",
                                  "region": "wb_region", "incomeLevel": "income_group"})
                 .loc[:, ["country", "country_name", "wb_region", "income_group"]])
    gdp = gdp[gdp["country"].isin(economies["country"])]
    print("World Bank API:", gdp.shape, economies.shape)
except Exception as e:
    print("Option A skipped:", e)

In [ ]:
if gdp is None:
    wide = pd.read_csv("data/raw/worldbank_gdp_pc_wide.csv", skiprows=4)
    wide = wide.loc[:, ~wide.columns.str.startswith("Unnamed")]
    year_cols = [c for c in wide.columns if c.isdigit()]
    gdp = (wide.melt(id_vars=["Country Name", "Country Code"], value_vars=year_cols, var_name="year", value_name="gdp_pc")
               .rename(columns={"Country Code": "country"})
               .assign(year=lambda d: d["year"].astype(int))
               .dropna(subset=["gdp_pc"])
               .loc[:, ["country", "year", "gdp_pc"]])
    meta = pd.read_csv("data/raw/worldbank_country_metadata.csv")
    economies = (meta.rename(columns={"Country Code": "country", "TableName": "country_name",
                                      "Region": "wb_region", "IncomeGroup": "income_group"})
                     .dropna(subset=["wb_region"])
                     .loc[:, ["country", "country_name", "wb_region", "income_group"]])
    gdp = gdp[gdp["country"].isin(economies["country"])]
    print("World Bank CSV:", gdp.shape, economies.shape)
print(gdp.head(3))
print(economies.head(3))

**Task 10.1**

Before merging, explore `gdp` and `economies`: (a) how many countries and which year range does `gdp`
cover? (b) Show GDP per capita of India (IND) in 1990, 2000 and 2010 (filter with `isin` on year).
(c) How many countries are in each `income_group`?

In [ ]:
# TASK 10.1

<details>
<summary>Solution 10.1</summary>

```python
print(gdp["country"].nunique(), gdp["year"].min(), gdp["year"].max())
print(gdp[(gdp["country"] == "IND") & (gdp["year"].isin([1990, 2000, 2010]))])
print(economies["income_group"].value_counts())
```

Output on the practice corpus:

```
36 1970 2018
     country  year  gdp_pc
736      IND  1990   718.0
1096     IND  2000   853.7
1456     IND  2010  1403.7
income_group
High income            9
Upper middle income    9
Lower middle income    9
Low income             9
Name: count, dtype: int64
```
</details>

### 10.2 Check keys, then merge

Keys must have identical types. The right table's key should be unique, or the merge multiplies rows.
`how="left"` keeps all speeches; `indicator=True` shows which rows matched.

In [ ]:
speeches["year"] = speeches["year"].astype(int)
gdp["year"] = gdp["year"].astype(int)
gdp["country"] = gdp["country"].str.upper().str.strip()
print("gdp key unique:", not gdp.duplicated(subset=["country", "year"]).any())

merged = pd.merge(speeches, gdp, on=["country", "year"], how="left", indicator=True)
print(merged["_merge"].value_counts())
print(len(speeches), len(merged))

In [ ]:
unmatched = (merged[merged["_merge"] == "left_only"]
             .groupby("country")["year"].agg(["min", "max", "size"]).sort_values("size", ascending=False))
print(unmatched.head())   # empty on the practice corpus; historical codes on the real one (YUG, CSK, DDR ...)
merged = merged.drop(columns="_merge")
merged = pd.merge(merged, economies, on="country", how="left")
merged["log_gdp_pc"] = np.log(merged["gdp_pc"])
df = merged.copy()
df.to_csv("data/processed/speeches_with_gdp.csv", index=False)
print(df.shape)

**Task 10.2**

Practise the merge logic on small tables. `left = pd.DataFrame({"country": ["BRA", "IND", "KEN", "YUG"],
"year": [2000, 2000, 2000, 1985], "words": [1000, 1200, 900, 800]})` and `right = pd.DataFrame({"country":
["BRA", "IND", "KEN", "JPN"], "year": [2000, 2000, 2001, 2000], "gdp_pc": [3700, 440, 400, 39000]})`.
(a) Left merge on country and year with `indicator=True`; print the result. Which rows did not match and
why? (b) Inner merge: how many rows? (c) Merge on `country` only: how many rows, and why is that wrong here?

In [ ]:
# TASK 10.2
left = pd.DataFrame({"country": ["BRA", "IND", "KEN", "YUG"], "year": [2000, 2000, 2000, 1985], "words": [1000, 1200, 900, 800]})
right = pd.DataFrame({"country": ["BRA", "IND", "KEN", "JPN"], "year": [2000, 2000, 2001, 2000], "gdp_pc": [3700, 440, 400, 39000]})

<details>
<summary>Solution 10.2</summary>

```python
left = pd.DataFrame({"country": ["BRA", "IND", "KEN", "YUG"], "year": [2000, 2000, 2000, 1985], "words": [1000, 1200, 900, 800]})
right = pd.DataFrame({"country": ["BRA", "IND", "KEN", "JPN"], "year": [2000, 2000, 2001, 2000], "gdp_pc": [3700, 440, 400, 39000]})
print(pd.merge(left, right, on=["country", "year"], how="left", indicator=True))
# KEN: year differs (2000 vs 2001); YUG: not in right at all
print(len(pd.merge(left, right, on=["country", "year"], how="inner")))
print(len(pd.merge(left, right, on="country", how="left")))
# merging on country only matches KEN 2000 with KEN 2001: wrong year, silently
```

Output on the practice corpus:

```
country  year  words  gdp_pc     _merge
0     BRA  2000   1000  3700.0       both
1     IND  2000   1200   440.0       both
2     KEN  2000    900     NaN  left_only
3     YUG  1985    800     NaN  left_only
2
4
```
</details>

**Task 10.3**

Now on `df`: (a) share of speeches with a non-missing `gdp_pc`, per decade (`groupby` then
`.apply(lambda s: s.notna().mean())`). (b) Mean `kw_development_pt` and `kw_climate_pt` by `income_group`,
rounded 2. (c) The five country-years with the lowest `gdp_pc` and their `kw_debt_pt`.

In [ ]:
# TASK 10.3

<details>
<summary>Solution 10.3</summary>

```python
print(df.groupby("decade")["gdp_pc"].apply(lambda s: s.notna().mean()).round(2))
print(df.groupby("income_group")[["kw_development_pt", "kw_climate_pt"]].mean().round(2))
print(df.nsmallest(5, "gdp_pc")[["country", "year", "gdp_pc", "kw_debt_pt"]])
```

Output on the practice corpus:

```
decade
1970s    1.0
1980s    1.0
1990s    1.0
2000s    1.0
2010s    1.0
Name: gdp_pc, dtype: float64
                     kw_development_pt  kw_climate_pt
income_group                                         
High income                       6.58           2.99
Low income                       13.68           3.02
Lower middle income              11.61           2.88
Upper middle income               9.36           2.92
    country  year  gdp_pc  kw_debt_pt
51      IDN  1971   248.3   14.400000
118     ETH  1973   282.3    6.887052
15      IDN  1970   284.8   16.997167
87      IDN  1972   290.7   13.994911
154     ETH  1974   291.8    3.378378
```
</details>

## Chapter 11: Time series

Lags, differences, growth, rolling means. With several countries, always `sort_values(["country", "year"])`
first and do every time operation inside `groupby("country")`.

In [ ]:
df = df.sort_values(["country", "year"]).reset_index(drop=True)
df["gdp_lag"] = df.groupby("country")["gdp_pc"].shift(1)
df["gdp_growth"] = df.groupby("country")["gdp_pc"].pct_change()
df["year_gap"] = df.groupby("country")["year"].diff()
df[df["country"] == "CHE"][["year", "gdp_pc", "gdp_lag", "gdp_growth", "year_gap"]].head(3).round(3)

**Task 11.1**

(a) Create `trade_lag1`, the previous year's `kw_trade_pt` within country, and `trade_change`, the difference
to it. Show Brazil's (BRA) first four rows of `year, kw_trade_pt, trade_lag1, trade_change`. (b) Which
country-year had the largest **drop** in `kw_debt_pt` from one year to the next? (c) Count the rows where
`year_gap` is greater than 1 (missing years in a country's series).

In [ ]:
# TASK 11.1

<details>
<summary>Solution 11.1</summary>

```python
df["trade_lag1"] = df.groupby("country")["kw_trade_pt"].shift(1)
df["trade_change"] = df["kw_trade_pt"] - df["trade_lag1"]
print(df[df["country"] == "BRA"][["year", "kw_trade_pt", "trade_lag1", "trade_change"]].head(4).round(2))
df["debt_change"] = df.groupby("country")["kw_debt_pt"].diff()
print(df.nsmallest(1, "debt_change")[["country", "year", "debt_change"]])
print((df["year_gap"] > 1).sum())
```

Output on the practice corpus:

```
year  kw_trade_pt  trade_lag1  trade_change
98   1970         0.97         NaN           NaN
99   1971         1.67        0.97          0.70
100  1972         2.65        1.67          0.98
101  1973         0.00        2.65         -2.65
     country  year  debt_change
1129     NGA  1972   -16.474944
0
```
</details>

In [ ]:
df["trade_smooth"] = df.groupby("country")["kw_trade_pt"].transform(
    lambda s: s.rolling(window=5, center=True, min_periods=3).mean())
df["climate_cum"] = df.groupby("country")["kw_climate"].cumsum()
df[df["country"] == "DEU"][["year", "kw_trade_pt", "trade_smooth", "climate_cum"]].tail(3).round(2)

**Task 11.2**

(a) Create `debt_roll10`, a 10-year rolling mean of `kw_debt_pt` within country (not centered, min 5
periods) and show Kenya's (KEN) last five rows of `year, kw_debt_pt, debt_roll10`. (b) Create `terror_cum`,
the cumulative count of `kw_terrorism` within country, and show which country has the highest final
value (`groupby("country")["terror_cum"].max().idxmax()`). (c) For each country, the first year in
which it mentioned climate; show the five earliest.

In [ ]:
# TASK 11.2

<details>
<summary>Solution 11.2</summary>

```python
df["debt_roll10"] = df.groupby("country")["kw_debt_pt"].transform(lambda s: s.rolling(10, min_periods=5).mean())
print(df[df["country"] == "KEN"][["year", "kw_debt_pt", "debt_roll10"]].tail().round(2))
df["terror_cum"] = df.groupby("country")["kw_terrorism"].cumsum()
print(df.groupby("country")["terror_cum"].max().idxmax())
print(df[df["mentions_climate"] == 1].groupby("country")["year"].min().sort_values().head())
```

Output on the practice corpus:

```
year  kw_debt_pt  debt_roll10
975  2014        4.74         5.22
976  2015        9.27         5.22
977  2016        6.41         5.40
978  2017        5.30         5.50
979  2018        4.50         5.15
MAR
country
KOR    1971
FRA    1972
PHL    1972
EGY    1972
GRC    1973
Name: year, dtype: int64
```
</details>

### 11.3 Real dates

`pd.to_datetime` parses dates; as an index they unlock `.dt` attributes and `resample`.
Frequencies: "YE" year end, "QE" quarter end, "ME" month end (older pandas: "Y", "Q", "M").

In [ ]:
d = pd.DataFrame({"date": ["2021-01-20", "2021-04-28", "2021-09-09", "2022-03-01"], "n_words": [2500, 6200, 3300, 7100]})
d["date"] = pd.to_datetime(d["date"])
print(d["date"].dt.year.tolist(), d["date"].dt.month.tolist())
print(d.set_index("date").resample("YE")["n_words"].sum())

**Task 11.3**

`e = pd.DataFrame({"date": ["2019-02-05", "2019-02-26", "2019-11-30", "2020-01-15", "2020-07-04"],
"n_words": [1800, 2400, 3100, 2900, 2200]})`. Convert `date`, add a `weekday` column with the day name
(`.dt.day_name()`), then compute the number of speeches and the total words per quarter with `resample("QE")`
and `agg(["count", "sum"])`.

In [ ]:
# TASK 11.3

<details>
<summary>Solution 11.3</summary>

```python
e = pd.DataFrame({"date": ["2019-02-05", "2019-02-26", "2019-11-30", "2020-01-15", "2020-07-04"],
                  "n_words": [1800, 2400, 3100, 2900, 2200]})
e["date"] = pd.to_datetime(e["date"])
e["weekday"] = e["date"].dt.day_name()
print(e)
print(e.set_index("date").resample("QE")["n_words"].agg(["count", "sum"]))
```

Output on the practice corpus:

```
date  n_words    weekday
0 2019-02-05     1800    Tuesday
1 2019-02-26     2400    Tuesday
2 2019-11-30     3100   Saturday
3 2020-01-15     2900  Wednesday
4 2020-07-04     2200   Saturday
            count   sum
date                   
2019-03-31      2  4200
2019-06-30      0     0
2019-09-30      0     0
2019-12-31      1  3100
2020-03-31      1  2900
2020-06-30      0     0
2020-09-30      1  2200
```
</details>

## Chapter 12: Plots

Pattern for every figure:

```python
fig, ax = plt.subplots(figsize=(w, h))
... plot onto ax ...
ax.set_xlabel(...); ax.set_ylabel(...); ax.set_title(...)
plt.show()
```

Labels with units and a title that states the finding. `fig.savefig("figures/name.png", dpi=150, bbox_inches="tight")` saves it.

In [ ]:
by_year = df.groupby("year")[["kw_trade_pt", "kw_debt_pt"]].mean()
fig, ax = plt.subplots(figsize=(8, 4))
by_year.plot(ax=ax)
ax.set_xlabel("Year")
ax.set_ylabel("Mentions per 1,000 words")
ax.set_title("Trade and debt over time, mean across countries")
ax.legend(["trade", "debt"])
plt.show()

**Task 12.1**

Plot the yearly mean of `kw_climate_pt` and `kw_terrorism_pt` as two lines on one axes, with axis labels,
a title and a legend reading "climate" and "terrorism". Save it as `figures/climate_terrorism.png`.

In [ ]:
# TASK 12.1

<details>
<summary>Solution 12.1</summary>

```python
by_year2 = df.groupby("year")[["kw_climate_pt", "kw_terrorism_pt"]].mean()
fig, ax = plt.subplots(figsize=(8, 4))
by_year2.plot(ax=ax)
ax.set_xlabel("Year")
ax.set_ylabel("Mentions per 1,000 words")
ax.set_title("Climate rises steadily, terrorism jumps after 2001")
ax.legend(["climate", "terrorism"])
fig.savefig("figures/climate_terrorism.png", dpi=150, bbox_inches="tight")
plt.show()
```

Output on the practice corpus:

```
[figure]
```
</details>

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
df.groupby("income_group")["kw_debt_pt"].mean().sort_values().plot.barh(ax=ax)
ax.set_xlabel("Debt mentions per 1,000 words")
ax.set_ylabel("")
ax.set_title("Debt talk by income group")
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df["log_gdp_pc"], df["kw_trade_pt"], s=8, alpha=0.3)
ax.set_xlabel("log GDP per capita")
ax.set_ylabel("Trade mentions per 1,000 words")
plt.show()

**Task 12.2**

(a) Horizontal bar chart of mean `kw_development_pt` by `wb_region`, sorted, with labels and title.
(b) Scatter of `log_gdp_pc` (x) against `kw_development_pt` (y) for speeches after 2000 only, small
semi-transparent points, labelled axes.

In [ ]:
# TASK 12.2

<details>
<summary>Solution 12.2</summary>

```python
fig, ax = plt.subplots(figsize=(6, 3.5))
df.groupby("wb_region")["kw_development_pt"].mean().sort_values().plot.barh(ax=ax)
ax.set_xlabel("Development mentions per 1,000 words")
ax.set_ylabel("")
ax.set_title("Development talk by region")
plt.show()

sub = df[df["year"] > 2000]
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(sub["log_gdp_pc"], sub["kw_development_pt"], s=8, alpha=0.3)
ax.set_xlabel("log GDP per capita")
ax.set_ylabel("Development mentions per 1,000 words")
ax.set_title("After 2000")
plt.show()
```

Output on the practice corpus:

```
[figure]
[figure]
```
</details>

### 12.3 Several lines by hand, small multiples

Loop over groups and call `ax.plot` for each. For panels, `plt.subplots(nrows, ncols)` returns an array of axes.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for c in ["CHE", "DEU", "USA"]:
    sub = df[df["country"] == c]
    ax.plot(sub["year"], sub["trade_smooth"], label=c)
ax.legend()
ax.set_xlabel("Year")
ax.set_ylabel("Trade per 1,000 words, 5-year smoothed")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for ax, dec in zip(axes, ["1980s", "2010s"]):
    s = df[df["decade"] == dec]
    ax.scatter(s["log_gdp_pc"], s["kw_debt_pt"], s=8, alpha=0.4)
    ax.set_title(dec)
fig.suptitle("Debt mentions vs income")
plt.show()

**Task 12.3**

(a) One axes with `kw_debt_pt` over time for Brazil, Mexico and Argentina (BRA, MEX, ARG), with a legend.
(b) A 2 by 2 grid of scatter plots of `log_gdp_pc` against `kw_climate_pt`, one panel per decade 1980s,
1990s, 2000s, 2010s (hint: `axes.flatten()`), with each panel titled by its decade and shared y axis.

In [ ]:
# TASK 12.3

<details>
<summary>Solution 12.3</summary>

```python
fig, ax = plt.subplots(figsize=(8, 4))
for c in ["BRA", "MEX", "ARG"]:
    sub = df[df["country"] == c]
    ax.plot(sub["year"], sub["kw_debt_pt"], label=c)
ax.legend()
ax.set_xlabel("Year")
ax.set_ylabel("Debt per 1,000 words")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharey=True)
for ax, dec in zip(axes.flatten(), ["1980s", "1990s", "2000s", "2010s"]):
    s = df[df["decade"] == dec]
    ax.scatter(s["log_gdp_pc"], s["kw_climate_pt"], s=8, alpha=0.4)
    ax.set_title(dec)
fig.suptitle("Climate mentions vs income")
fig.tight_layout()
plt.show()
```

Output on the practice corpus:

```
[figure]
[figure]
```
</details>

## Chapter 13: Statistics

`corr` in pandas; regressions with `statsmodels`, formula syntax like R. `C(x)` makes dummies, which is how
you get fixed effects. Cluster standard errors by country for panel data.

In [ ]:
import statsmodels.formula.api as smf

print(df[["log_gdp_pc", "kw_trade_pt", "kw_debt_pt", "n_words"]].corr().round(2))
reg = df.dropna(subset=["log_gdp_pc", "kw_debt_pt"])
m1 = smf.ols("kw_debt_pt ~ log_gdp_pc", data=reg).fit()
print(m1.params.round(3))
m2 = smf.ols("kw_debt_pt ~ log_gdp_pc + C(year) + C(country)", data=reg).fit(
    cov_type="cluster", cov_kwds={"groups": reg["country"]})
print("two-way FE:", round(m2.params["log_gdp_pc"], 3), "se", round(m2.bse["log_gdp_pc"], 3))

**Task 13.1**

(a) Correlation between `log_gdp_pc` and `kw_development_pt`, overall and within the 1990s only.
(b) Pooled OLS of `kw_development_pt` on `log_gdp_pc`: print the coefficient and its p-value
(`m.pvalues["log_gdp_pc"]`). (c) The same with year and country fixed effects and clustered standard
errors. (d) Add `n_words` as a control to (c). Does the coefficient on `log_gdp_pc` change much?

In [ ]:
# TASK 13.1

<details>
<summary>Solution 13.1</summary>

```python
print(df[["log_gdp_pc", "kw_development_pt"]].corr().iloc[0, 1].round(3))
print(df[df["decade"] == "1990s"][["log_gdp_pc", "kw_development_pt"]].corr().iloc[0, 1].round(3))
r = df.dropna(subset=["log_gdp_pc", "kw_development_pt"])
ma = smf.ols("kw_development_pt ~ log_gdp_pc", data=r).fit()
print(round(ma.params["log_gdp_pc"], 3), round(ma.pvalues["log_gdp_pc"], 4))
mb = smf.ols("kw_development_pt ~ log_gdp_pc + C(year) + C(country)", data=r).fit(
    cov_type="cluster", cov_kwds={"groups": r["country"]})
print(round(mb.params["log_gdp_pc"], 3), round(mb.bse["log_gdp_pc"], 3))
mc = smf.ols("kw_development_pt ~ log_gdp_pc + n_words + C(year) + C(country)", data=r).fit(
    cov_type="cluster", cov_kwds={"groups": r["country"]})
print(round(mc.params["log_gdp_pc"], 3))
```

Output on the practice corpus:

```
-0.556
-0.576
-2.029 0.0
-0.372 0.652
-0.275
```
</details>

### 13.2 Loop a regression over outcomes and collect results in a DataFrame

A list of dictionaries becomes a results table. This is how you produce a regression table for slides.

In [ ]:
rows = []
for topic in ["trade", "debt"]:
    y = f"kw_{topic}_pt"
    s = df.dropna(subset=["log_gdp_pc", y])
    m = smf.ols(f"{y} ~ log_gdp_pc + C(year) + C(country)", data=s).fit(
        cov_type="cluster", cov_kwds={"groups": s["country"]})
    rows.append({"topic": topic, "coef": m.params["log_gdp_pc"], "se": m.bse["log_gdp_pc"], "n": int(m.nobs)})
pd.DataFrame(rows).set_index("topic").round(3)

**Task 13.2**

Do the same for all six topics in `keywords`, but with year fixed effects only (no country FE) and plain
standard errors. Add a column `p` with the p-value. Sort the table by `coef`. Then, in a comment, write one
sentence on why the year fixed effects matter so much for the climate topic.

In [ ]:
# TASK 13.2

<details>
<summary>Solution 13.2</summary>

```python
rows = []
for topic in keywords:
    y = f"kw_{topic}_pt"
    s = df.dropna(subset=["log_gdp_pc", y])
    m = smf.ols(f"{y} ~ log_gdp_pc + C(year)", data=s).fit()
    rows.append({"topic": topic, "coef": m.params["log_gdp_pc"], "se": m.bse["log_gdp_pc"],
                 "p": m.pvalues["log_gdp_pc"], "n": int(m.nobs)})
print(pd.DataFrame(rows).set_index("topic").sort_values("coef").round(3))
# Climate rises for everyone over time while GDP also rises, so without year FE the two trends
# are confounded; year FE compare countries within the same year.
```

Output on the practice corpus:

```
coef     se      p     n
topic                                 
development -2.053  0.075  0.000  1764
debt        -1.454  0.063  0.000  1764
terrorism   -0.018  0.022  0.413  1764
climate     -0.004  0.040  0.925  1764
peace        0.132  0.047  0.005  1764
trade        0.739  0.058  0.000  1764
```
</details>

## Chapter 14: Word frequencies

Tokenize with `str.findall`, `explode` to one row per word, `value_counts`. Remove stopwords. Then TF-IDF
to find words that characterize a group.

In [ ]:
tokens = df["text_lower"].str.findall(r"\b[a-z]{3,}\b")
words = df[["country", "year", "decade", "income_group"]].assign(word=tokens).explode("word")
stopwords = set("""the of and to in that is for we our with this on by be are it as has have from which at not
will all its an or was their these those been can they also must should would there but one more than
his her he she who what when where how into such only other some them very about between through my your
a i you if so no do does did had were being above under over after before""".split())
words = words[~words["word"].isin(stopwords)]
print(words.shape)
print(words["word"].value_counts().head(5))

**Task 14.1**

(a) The ten most frequent words in speeches by low-income countries ("Low income") after 2005.
(b) The five most frequent words in Japan's (JPN) speeches. (c) How many distinct words does the whole
corpus use (`nunique`)? (d) Add `vocab_size` to `df`: the number of distinct words per speech
(`tokens.apply(lambda t: len(set(t)))`), and show the country with the highest mean vocabulary size.

In [ ]:
# TASK 14.1

<details>
<summary>Solution 14.1</summary>

```python
w = words[(words["income_group"] == "Low income") & (words["year"] > 2005)]
print(w["word"].value_counts().head(10))
print(words[words["country"] == "JPN"]["word"].value_counts().head())
print(words["word"].nunique())
df["vocab_size"] = tokens.apply(lambda t: len(set(t)))
print(df.groupby("country")["vocab_size"].mean().idxmax())
```

Output on the practice corpus:

```
word
nations          1324
international    1204
remains          1122
security         1119
government       1112
attention        1069
development       769
debt              755
call              731
community         578
Name: count, dtype: int64
word
nations          576
international    549
government       528
remains          527
security         517
Name: count, dtype: int64
129
PHL
```
</details>

In [ ]:
top_by_decade = (words.groupby("decade")["word"].value_counts()
                      .groupby(level=0).head(3))
print(top_by_decade)

from sklearn.feature_extraction.text import TfidfVectorizer
docs = df.groupby("decade")["text_lower"].apply(" ".join)
vec = TfidfVectorizer(stop_words="english", max_features=5000, token_pattern=r"\b[a-z]{3,}\b")
X = vec.fit_transform(docs)
tfidf = pd.DataFrame(X.toarray(), index=docs.index, columns=vec.get_feature_names_out())
for dec in tfidf.index:
    print(dec, ":", ", ".join(tfidf.loc[dec].sort_values(ascending=False).head(5).index))

On the practice corpus the TF-IDF lists look alike, because all decades draw from the same sentence pool.
On the real corpus they differ and are a good source of words for the keyword dictionary.

**Task 14.2**

(a) Top three words per `income_group` with the groupby + value_counts + `head` trick. (b) TF-IDF with
`wb_region` as documents (drop rows with a missing region first); print the top five words per region.
(c) Relative frequency: compute the share of each word within "High income" speeches and within all
speeches, merge the two, and show the five words with the highest ratio among words with an overall share
above 0.002 (hint: `value_counts(normalize=True)`, `.rename`, `.reset_index()`, `pd.merge` on `word`).

In [ ]:
# TASK 14.2

<details>
<summary>Solution 14.2</summary>

```python
print(words.groupby("income_group")["word"].value_counts().groupby(level=0).head(3))

docs_r = df.dropna(subset=["wb_region"]).groupby("wb_region")["text_lower"].apply(" ".join)
Xr = vec.fit_transform(docs_r)
tr = pd.DataFrame(Xr.toarray(), index=docs_r.index, columns=vec.get_feature_names_out())
for r_ in tr.index:
    print(r_, ":", ", ".join(tr.loc[r_].sort_values(ascending=False).head(5).index))

hi = words[words["income_group"] == "High income"]["word"].value_counts(normalize=True).rename("hi_share").reset_index()
allw = words["word"].value_counts(normalize=True).rename("all_share").reset_index()
rel = pd.merge(hi, allw, on="word")
rel["ratio"] = rel["hi_share"] / rel["all_share"]
print(rel[rel["all_share"] > 0.002].sort_values("ratio", ascending=False).head().round(4))
```

Output on the practice corpus:

```
income_group         word         
High income          nations          5539
                     international    5295
                     attention        4974
Low income           nations          4812
                     international    4626
                     government       4542
Lower middle income  nations          5200
                     international    4768
                     security         4705
Upper middle income  nations          5246
                     international    5066
                     government       4865
Name: count, dtype: int64
Africa : nations, international, government, security, remains
Americas : nations, international, government, security, remains
Asia : nations, international, security, government, remains
Europe : nations, international, attention, security, government
Oceania : nations, international, attention, remains, security
             word  hi_share  all_share   ratio
66          trade    0.0063     0.0047  1.3242
1   international    0.0265     0.0246  1.0787
7           thank    0.0126     0.0117  1.0770
8        assembly    0.0126     0.0117  1.0770
12        resolve    0.0124     0.0115  1.0729
```
</details>

## Chapter 15: Your project

No examples here, only tasks. They turn the exploration above into a pipeline and a first result section.
Use everything from Chapters 3 to 13. Solutions are given, but they are one way of many.

**Task 15.1**

Write four functions and run them as a pipeline:
- `clean_speeches(raw_df)`: types, drop missing, upper-case codes, clean text, drop duplicate country-years
  keeping the longest, add `text_lower`, `n_words`, `decade`; return sorted by country and year.
- `add_keywords(clean_df, keyword_dict)`: the three columns per topic.
- `add_economics(kw_df, gdp_table, economies_table)`: the two merges and `log_gdp_pc`.
- `topic_table(final_df, topic)`: returns a DataFrame with the mean per-thousand of `topic` by
  `income_group` and `decade` (pivot table).

Then: `final = add_economics(add_keywords(clean_speeches(raw), keywords), gdp, economies)` and print
`topic_table(final, "debt").round(2)`.

In [ ]:
# TASK 15.1

<details>
<summary>Solution 15.1</summary>

```python
def clean_speeches(raw_df):
    out = raw_df.copy()
    out["year"] = pd.to_numeric(out["year"], errors="coerce")
    out = out.dropna(subset=["text", "country", "year"])
    out["year"] = out["year"].astype(int)
    out["country"] = out["country"].str.strip().str.upper()
    out["text"] = (out["text"].str.replace("\ufeff", "", regex=False)
                              .str.replace(r"\s+", " ", regex=True).str.strip())
    out["text_len"] = out["text"].str.len()
    out = (out.sort_values("text_len", ascending=False)
              .drop_duplicates(subset=["country", "year"]).drop(columns="text_len"))
    out["text_lower"] = out["text"].str.lower()
    out["n_words"] = out["text"].str.split().str.len()
    out["decade"] = (out["year"] // 10 * 10).astype(str) + "s"
    return out.sort_values(["country", "year"]).reset_index(drop=True)

def add_keywords(clean_df, keyword_dict):
    out = clean_df.copy()
    for topic, pattern in keyword_dict.items():
        out[f"kw_{topic}"] = out["text_lower"].str.count(pattern)
        out[f"kw_{topic}_pt"] = 1000 * out[f"kw_{topic}"] / out["n_words"]
        out[f"mentions_{topic}"] = (out[f"kw_{topic}"] > 0).astype(int)
    return out

def add_economics(kw_df, gdp_table, economies_table):
    out = pd.merge(kw_df, gdp_table, on=["country", "year"], how="left")
    out = pd.merge(out, economies_table, on="country", how="left")
    out["log_gdp_pc"] = np.log(out["gdp_pc"])
    return out

def topic_table(final_df, topic):
    return pd.pivot_table(final_df, values=f"kw_{topic}_pt", index="income_group", columns="decade", aggfunc="mean")

final = add_economics(add_keywords(clean_speeches(raw), keywords), gdp, economies)
print(final.shape)
print(topic_table(final, "debt").round(2))
```

Output on the practice corpus:

```
(1764, 30)
decade               1970s  1980s  1990s  2000s  2010s
income_group                                          
High income           3.81   7.19   4.66   2.70   2.40
Low income            8.86  11.59   9.43   7.24   7.83
Lower middle income   7.27  10.28   9.38   6.55   6.32
Upper middle income   5.60   9.04   6.30   4.26   4.53
```
</details>

**Task 15.2**

Produce a first "results" block for a presentation, using `final`:
1. A coverage table: speeches per decade and share with GDP data.
2. The figure: mean per-thousand of all six topics by year, one line each, saved to `figures/topics.png`.
3. The regression table: two-way FE with clustered SE, one row per topic, columns coef, se, p, n.
4. A three-sentence interpretation in a comment, including the causal caveat.

In [ ]:
# TASK 15.2

<details>
<summary>Solution 15.2</summary>

```python
cov = final.groupby("decade").agg(n=("year", "size"), gdp_share=("gdp_pc", lambda s: s.notna().mean())).round(2)
print(cov)

by_year_all = final.groupby("year")[[f"kw_{t}_pt" for t in keywords]].mean()
fig, ax = plt.subplots(figsize=(9, 4.5))
by_year_all.plot(ax=ax)
ax.set_xlabel("Year"); ax.set_ylabel("Mentions per 1,000 words")
ax.set_title("Topic intensity in UN General Debate speeches")
ax.legend(list(keywords))
fig.savefig("figures/topics.png", dpi=150, bbox_inches="tight")
plt.show()

rows = []
for topic in keywords:
    y = f"kw_{topic}_pt"
    s = final.dropna(subset=["log_gdp_pc", y])
    m = smf.ols(f"{y} ~ log_gdp_pc + C(year) + C(country)", data=s).fit(
        cov_type="cluster", cov_kwds={"groups": s["country"]})
    rows.append({"topic": topic, "coef": m.params["log_gdp_pc"], "se": m.bse["log_gdp_pc"],
                 "p": m.pvalues["log_gdp_pc"], "n": int(m.nobs)})
print(pd.DataFrame(rows).set_index("topic").round(3))
# Within a country over time, higher income goes with less debt talk and about as much trade talk.
# Year effects absorb the global rise of climate and the 2001 jump in terrorism.
# These are conditional correlations: income and rhetoric both respond to crises and politics,
# so no causal reading is justified.
```

Output on the practice corpus:

```
n  gdp_share
decade                
1970s   360        1.0
1980s   360        1.0
1990s   360        1.0
2000s   360        1.0
2010s   324        1.0
[figure]
              coef     se      p     n
topic                                 
trade        0.384  0.705  0.586  1764
debt        -1.423  0.576  0.013  1764
development -0.372  0.652  0.569  1764
climate     -0.142  0.354  0.688  1764
terrorism    0.196  0.261  0.453  1764
peace       -0.534  0.387  0.167  1764
```
</details>

### Next steps

- Replace the practice corpus with the real Kaggle file and rerun from the top. Study the unmatched
  country codes in Chapter 10 and add a recoding dictionary.
- Move the four functions from 15.1 into `speech_tools.py` and import them.
- Pick one question with a second World Bank indicator (trade openness `NE.TRD.GNFS.ZS`, external debt
  `DT.DOD.DECT.GN.ZS`) and build the merge, figure and regression for it.
- Commit after each session: `git add .`, `git commit -m "..."`, `git push`.